# AI Workforce Statements: Complete Revelio Firm Matching

## Part 0 Notebook Setup and Research Scope

**Purpose.** Extend the **provisional** historical firm crosswalk prepared in
`01_build_firm_crosswalk.ipynb` by resolving remaining Revelio company identities
for firms represented in the S&P 500 from **January 1, 2018 through December 31, 2025**.

This notebook is a follow-up to Notebook 01, **not a replacement** for its CRSP–CCM
historical linking and Compustat candidate universe. The last saved baseline reported:

- **652** candidate GVKEY firms.
- **539** firms with direct Revelio GVKEY candidates.
- **26** additional firms with local CIK and/or full-CUSIP candidates.
- **87** firms without any candidate in the *ticker-filtered* Revelio extract.

These counts describe **candidate availability**, not 565 accepted entity matches.
The exported Notebook 01 crosswalk is provisional; RCIDs require explicit review.

### Sources and units of observation

| Source | Unit of observation | Relevant identifiers |
|---|---|---|
| Notebook 01 `crosswalk.parquet` | One provisional GVKEY firm | `firm_id`, GVKEY, candidate identity/status |
| Notebook 01 `crosswalk_membership_bridge.parquet` | Historical membership interval | PERMNO, PERMCO, dated CCM evidence |
| Compustat Fundamentals Annual | Annual company observation | GVKEY, CIK, CUSIP, ticker, company name |
| Existing Revelio mapping | Ticker-filtered company candidates | RCID, CIK, CUSIP, ticker, name, LinkedIn URL |
| **Full Revelio company reference (downloaded)** | Revelio company record | RCID and available identifiers / hierarchy |

**Workflow.** Preserve Notebook 01's historical target universe. Discover candidates
independently by exact GVKEY, normalized positive CIK, and validated full CUSIP
against the full reference; integrate all earlier candidates; assess evidence and
ambiguity; prepare researcher-reviewed deliverables. Earlier ticker/name evidence
is supplementary. Discovery coverage is never accepted-match coverage.

**Identifier integrity.** Keep GVKEY, CIK, CUSIP, ticker, and RCID as strings where
appropriate. Preserve leading zeros. Distinguish issue-level CUSIP from company identity.
Never overwrite original source identifiers with normalized comparison keys.

**Data handling.** Licensed company-level data and review tables remain local under
`data/`. Preserve the Notebook 01 outputs. No original source file or v1 crosswalk
is modified. Review cell outputs before sharing this notebook publicly.

**Status at the 2026-11-08 edit.** Saved 2026-11-06 execution reports 651/652 firms
with full-reference candidates, 780 pairs, 115 multi-RCID firms, and 2,001 evidence
records from 33,961,589 scanned rows. The revised Part 2.2 below is **unexecuted**.
Part 2.3 awaits researcher verification of Part 2.2; final exports await separate
authorization. No identities have been accepted.

**Execution checkpoint.** Keep the completed kernel alive. For this iteration run
only **2.1.0**, then **2.2.1–2.2.5**. Do not use Run All. Full scans are disabled
by default. After creating a checkpoint, a fresh kernel can run Part 0 and 2.1.0
with the explicit checkpoint path, then Part 2.2 directly. Without a live kernel
or checkpoint, the complete scan evidence cannot be restored from display output.

The original notebook, including the old 73-pair audit and all saved outputs, is
preserved verbatim at `output/revelio_workflow_archive/2026-11-08/01b_before_consolidation.ipynb`.
The redesign and future output schemas are documented in
`docs/revelio_redesign_2026-11-08.md`. New code uses `revelio_consolidation.py`
beside this notebook; imports perform no research I/O.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

study_start = pd.Timestamp("2018-01-01")
study_end = pd.Timestamp("2025-12-31")

# Same project-root convention used by Notebook 01.
for project_root in [Path("."), Path(".."), Path("../..")]:
    if (project_root / "data" / "raw").is_dir():
        break
else:
    raise FileNotFoundError("Run from the project root, code/, or code/notebooks/.")

print(f"Study period: {study_start.date()} to {study_end.date()}")
print(f"Project root: {project_root.resolve()}")
print(f"pandas version: {pd.__version__}")

Study period: 2018-01-01 to 2025-12-31
Project root: C:\Users\KPdesktop\Desktop\AI Workforce Statements
pandas version: 3.0.1


## Part 1 Input Inventory and Baseline Verification

### 1.1 Required inputs and availability

Check the existing v1 outputs and reference files **without loading or modifying** them.
The filename and location of the full Revelio company list are intentionally left
unassigned until the new extract is available. Do not mistake the old ticker-filtered
mapping for the full company universe.

In [2]:
input_paths = {
    "crosswalk_v1": project_root / "data/clean/crosswalk.parquet",
    "membership_bridge_v1": project_root / "data/clean/crosswalk_membership_bridge.parquet",
    "compustat": project_root / "data/raw/compustat/Compustat Fundamentals Annual.csv",
    "revelio_ticker_filtered": project_root / "data/raw/revelio/revelio_company_mapping_20261023.csv",
}

# Set only after obtaining and inspecting the FULL company-reference export.
revelio_full_company_path = None

inventory = pd.DataFrame([
    {
        "input": name,
        "relative_path": path.relative_to(project_root).as_posix(),
        "exists": path.is_file(),
        "size_mb": round(path.stat().st_size / 1_000_000, 2) if path.is_file() else pd.NA,
    }
    for name, path in input_paths.items()
])
display(inventory)

print("Full Revelio company reference:",
      "NOT CONFIGURED" if revelio_full_company_path is None else revelio_full_company_path)
# Inventory only: no dataframes loaded, no identities selected, no files written.

,input,relative_path,exists,size_mb
0,crosswalk_v1,data/clean/crosswalk.parquet,True,0.09
1,membership_bridge_v1,data/clean/crosswalk_membership_bridge.parquet,True,0.11
2,compustat,data/raw/compustat/Compustat Fundamentals Annu...,True,10.85
3,revelio_ticker_filtered,data/raw/revelio/revelio_company_mapping_20261...,True,0.25


Full Revelio company reference: NOT CONFIGURED


### 1.2 Baseline checks

Read the provisional crosswalk, Compustat, and the **existing ticker-filtered** Revelio
extract only. All reconstructed evidence stays in memory; no source or crosswalk export
is written. The saved GVKEY universe is fixed for this audit.

#### 1.2.1 Saved baseline and identifier completeness

Retain the loaded tables unchanged. CSV identifiers use string dtype with only empty
fields treated as missing (`NA` can be a ticker). Comparison keys are separate columns.

In [3]:
import json

crosswalk_v1 = pd.read_parquet(input_paths["crosswalk_v1"])
compustat_source = pd.read_csv(
    input_paths["compustat"], dtype="string", keep_default_na=False, na_values=[""],
)
revelio_local_source = pd.read_csv(
    input_paths["revelio_ticker_filtered"], dtype="string", keep_default_na=False, na_values=[""],
)
assert len(crosswalk_v1) == 652, "Saved firm baseline changed; investigate before proceeding."
assert crosswalk_v1["gvkey"].notna().all() and crosswalk_v1["gvkey"].is_unique
assert crosswalk_v1["firm_id"].is_unique
assert crosswalk_v1["firm_id"].eq("gvkey:" + crosswalk_v1["gvkey"]).all()
assert crosswalk_v1["gvkey"].str.fullmatch(r"[0-9]{6}").all()

completeness_rows = []
for column in ["gvkey", "conm", "company_name", "cik", "cusip", "cusip8", "ticker", "rcid"]:
    exists = column in crosswalk_v1
    present = (crosswalk_v1[column].astype("string").str.strip().replace("", pd.NA).notna().sum()
               if exists else 0)
    completeness_rows.append({"field": column, "in_schema": exists,
                              "present_firms": int(present), "missing_firms": 652 - int(present)})
display(pd.DataFrame(completeness_rows))
display(crosswalk_v1.groupby(["rcid_status", "method_category"], dropna=False)
        .size().rename("firms").to_frame())
print("Accepted RCIDs:", crosswalk_v1["rcid"].notna().sum())
assert crosswalk_v1["rcid"].isna().all(), "Saved acceptance state changed."
assert crosswalk_v1["decision_basis"].isna().all()

,field,in_schema,present_firms,missing_firms
0,gvkey,True,652,0
1,conm,False,0,652
2,company_name,False,0,652
3,cik,True,0,652
4,cusip,False,0,652
5,cusip8,True,0,652
6,ticker,True,0,652
7,rcid,True,0,652


firms
rcid_status          method_category                     
no_local_candidate   No local candidate                87
unreviewed_candidate Additional CIK and full CUSIP     25
                     Additional full CUSIP only         1
                     Direct GVKEY candidate           539

Accepted RCIDs: 0


#### 1.2.2 Reconstruct Compustat evidence and diagnose missing scalars

Notebook 01 Part 4 uses **all downloaded annual rows** for the candidate GVKEYs;
no year, format, or membership-date filter is applied to identifier evidence. Keep
source-row numbers, dates, formats, raw identifiers, and every distinct observed value.
Do not choose a latest or preferred annual observation.

Notebook 01 Part 7.2 aggregates tuples, then accepts a scalar only when
`isinstance(values, tuple)` and its length is one. The probe below tests that exact
aggregation on this pandas version. A list returned by aggregation fails this guard,
even when it contains exactly one identifier. Company name and full CUSIP were not
included in the final schema; `cusip8` was a derived prefix, not the original CUSIP.

In [4]:
compustat_evidence = compustat_source.copy()
compustat_evidence.columns = compustat_evidence.columns.str.lower()
compustat_evidence["compustat_row"] = range(1, len(compustat_evidence) + 1)
compustat_evidence["gvkey_key"] = compustat_evidence["gvkey"].str.strip().replace("", pd.NA)
compustat_evidence = compustat_evidence.loc[
    compustat_evidence["gvkey_key"].isin(crosswalk_v1["gvkey"])
].copy()
for column in ["conm", "cik", "cusip", "tic"]:
    compustat_evidence[column + "_key"] = compustat_evidence[column].str.strip().replace("", pd.NA)
# Notebook 01 Part 5: positive digits-only CIK, with zeros stripped ONLY in comparison keys.
cik_digits = compustat_evidence["cik_key"].str.fullmatch(r"[0-9]+").fillna(False)
cik_comparison = compustat_evidence["cik_key"].str.lstrip("0").replace("", "0")
compustat_evidence["cik_for_comparison"] = cik_comparison.where(cik_digits.astype(bool) & cik_comparison.ne("0").fillna(False).astype(bool))
cusip_comparison = compustat_evidence["cusip_key"].str.upper()
compustat_evidence["cusip_for_comparison"] = cusip_comparison.where(
    cusip_comparison.str.fullmatch(r"[0-9A-Z*@#]{8,9}").fillna(False)
)
compustat_evidence["cusip8_observed"] = compustat_evidence["cusip_for_comparison"].str[:8]

# Explicit records keep tuple containers independent of pandas GroupBy dtype coercion.
evidence_rows = []
for gvkey, annual in compustat_evidence.groupby("gvkey_key", sort=True):
    record = {"gvkey": gvkey, "annual_rows": len(annual)}
    for column in ["conm", "cik", "cusip", "tic", "cik_for_comparison", "cusip8_observed"]:
        values = annual[column]
        present = values.notna() & values.str.strip().ne("").fillna(False)
        record[column + "_values"] = tuple(sorted(values.loc[present].unique()))
    evidence_rows.append(record)
firm_evidence = crosswalk_v1[["gvkey"]].merge(
    pd.DataFrame(evidence_rows), on="gvkey", how="left", validate="one_to_one",
)
for column in firm_evidence.columns[2:]:
    firm_evidence[column] = [value if isinstance(value, tuple) else () for value in firm_evidence[column]]
firm_evidence["annual_rows"] = firm_evidence["annual_rows"].fillna(0).astype("int64")

evidence_completeness = pd.DataFrame([
    {"evidence": column, "firms_with_value": int(firm_evidence[column].map(len).gt(0).sum()),
     "firms_missing": int(firm_evidence[column].map(len).eq(0).sum()),
     "firms_multiple_values": int(firm_evidence[column].map(len).gt(1).sum())}
    for column in ["conm_values", "cik_values", "cusip_values", "tic_values"]
])
display(evidence_completeness)
print("Compustat annual rows:", len(compustat_evidence))
print("Firms without Compustat rows:", firm_evidence["annual_rows"].eq(0).sum())

scalar_diagnosis = []
for source, target in [("cik_for_comparison", "cik"), ("cusip8_observed", "cusip8"), ("tic_key", "ticker")]:
    probe = compustat_evidence.groupby("gvkey_key").agg(
        values=(source, lambda values: tuple(sorted(values.dropna().unique())))
    )["values"]
    scalar_diagnosis.append({
        "saved_field": target,
        "aggregation_container_types": ", ".join(sorted({type(value).__name__ for value in probe})),
        "single_value_groups": int(probe.map(len).eq(1).sum()),
        "pass_original_tuple_guard": int(probe.map(lambda value: isinstance(value, tuple) and len(value) == 1).sum()),
        "saved_nonnull": int(crosswalk_v1[target].notna().sum()),
    })
display(pd.DataFrame(scalar_diagnosis))
# Raw strings retain source padding; these counts do not normalize or replace them.
display(pd.Series({column: int(compustat_evidence[column].str.startswith("0").fillna(False).sum())
                   for column in ["gvkey", "cik", "cusip"]}, name="source_rows_with_leading_zero").to_frame())

,evidence,firms_with_value,firms_missing,firms_multiple_values
0,conm_values,640,12,0
1,cik_values,639,13,0
2,cusip_values,640,12,0
3,tic_values,640,12,0


Compustat annual rows: 5725
Firms without Compustat rows: 12


,saved_field,aggregation_container_types,single_value_groups,pass_original_tuple_guard,saved_nonnull
0,cik,list,639,0,0
1,cusip8,list,640,0,0
2,ticker,list,640,0,0


,source_rows_with_leading_zero
gvkey,4430
cik,5709
cusip,773


#### 1.2.3 Reproduce original local candidate definitions

Use Notebook 01 Parts 5?6: a nonblank RCID with an exact trimmed GVKEY, a positive
digits-only CIK equal after zero normalization, or an exact **full** valid 8/9-character
CUSIP after uppercasing. Never match CUSIP prefixes, tickers, or names in this baseline.
All observed Compustat identifiers participate, and conflicting Revelio GVKEYs do not
exclude supplementary candidates. Direct GVKEY takes precedence in the category only.

Compare every reconstructed `(GVKEY, RCID, method)` against the saved candidate JSON,
not merely the aggregate counts. No candidate is accepted.

In [5]:
revelio_local = revelio_local_source.copy()
revelio_local.columns = revelio_local.columns.str.lower()
for column in ["gvkey", "rcid", "cik", "cusip"]:
    revelio_local[column + "_key"] = revelio_local[column].str.strip().replace("", pd.NA)
r_cik = revelio_local["cik_key"].str.lstrip("0").replace("", "0")
revelio_local["cik_for_comparison"] = r_cik.where(
    revelio_local["cik_key"].str.fullmatch(r"[0-9]+").fillna(False).astype(bool)
    & r_cik.ne("0").fillna(False).astype(bool)
)
r_cusip = revelio_local["cusip_key"].str.upper()
revelio_local["cusip_for_comparison"] = r_cusip.where(
    r_cusip.str.fullmatch(r"[0-9A-Z*@#]{8,9}").fillna(False)
)
local_with_rcid = revelio_local.loc[revelio_local["rcid_key"].notna()]
direct = local_with_rcid.loc[local_with_rcid["gvkey_key"].isin(crosswalk_v1["gvkey"])]
reconstructed_candidates = {(row.gvkey_key, row.rcid_key, "direct_gvkey")
                            for row in direct.itertuples(index=False)}
for key, method in [("cik_for_comparison", "cik"), ("cusip_for_comparison", "full_cusip")]:
    left = compustat_evidence.loc[compustat_evidence[key].notna(), ["gvkey_key", key]].drop_duplicates()
    right = local_with_rcid.loc[local_with_rcid[key].notna(), ["rcid_key", key]].drop_duplicates()
    matched = left.merge(right, on=key, how="inner")
    reconstructed_candidates.update((row.gvkey_key, row.rcid_key, method)
                                    for row in matched.itertuples(index=False))

saved_candidates = set()
for row in crosswalk_v1.itertuples(index=False):
    for candidate in json.loads(row.revelio_candidates_json):
        saved_candidates.update((row.gvkey, candidate["rcid"], method) for method in candidate["methods"])
print("Reconstructed / saved candidate-method triples:", len(reconstructed_candidates), len(saved_candidates))
print("Added / missing triples:", len(reconstructed_candidates - saved_candidates),
      len(saved_candidates - reconstructed_candidates))
assert reconstructed_candidates == saved_candidates, "Local evidence differs from v1; investigate."

candidate_status = crosswalk_v1[["gvkey"]].copy()
for method in ["direct_gvkey", "cik", "full_cusip"]:
    keys = {gvkey for gvkey, rcid, evidence_method in reconstructed_candidates if evidence_method == method}
    candidate_status[method] = candidate_status["gvkey"].isin(keys)
candidate_status["candidate_status"] = "No local candidate"
no_direct = ~candidate_status["direct_gvkey"]
for cik_flag, cusip_flag, label in [
    (True, False, "Additional CIK only"),
    (False, True, "Additional full CUSIP only"),
    (True, True, "Additional CIK and full CUSIP"),
]:
    mask = no_direct & candidate_status["cik"].eq(cik_flag) & candidate_status["full_cusip"].eq(cusip_flag)
    candidate_status.loc[mask, "candidate_status"] = label
candidate_status.loc[candidate_status["direct_gvkey"], "candidate_status"] = "Direct GVKEY candidate"
assert candidate_status["candidate_status"].eq(crosswalk_v1["method_category"]).all()
no_local = candidate_status["candidate_status"].eq("No local candidate")
assert no_local.eq(crosswalk_v1["no_local_candidate"]).all()
assert no_local.eq(crosswalk_v1["rcid_status"].eq("no_local_candidate")).all()
assert int(candidate_status["direct_gvkey"].sum()) == 539
assert int((no_direct & ~no_local).sum()) == 26
assert int(no_local.sum()) == 87
display(candidate_status.groupby("candidate_status").size().rename("firms").to_frame())
assert len(saved_candidates) == len(reconstructed_candidates) == 1638
print("Verified all 1,638 saved candidate-method triples; no candidate accepted.")


Reconstructed / saved candidate-method triples: 1638 1638
Added / missing triples: 0 0


,firms
candidate_status,
Additional CIK and full CUSIP,25
Additional full CUSIP only,1
Direct GVKEY candidate,539
No local candidate,87


Verified all 1,638 saved candidate-method triples; no candidate accepted.


#### 1.2.4 Review table: firms without local candidates

`baseline_review` contains all 652 firms; `no_local_candidate_review` displays all 87
without local candidates. Identifier columns are tuples of **original Compustat values**,
including leading zeros; multiple values remain visible and `()` means no source evidence.
These are observed reference values, not verified historical identifiers. Source-row,
report-date, and format details remain available in `compustat_evidence`.

In [6]:
baseline_review = firm_evidence.merge(
    candidate_status[["gvkey", "candidate_status"]], on="gvkey", validate="one_to_one",
).rename(columns={"conm_values": "company_name", "cik_values": "cik",
                  "cusip_values": "cusip", "tic_values": "ticker"})
review_columns = ["gvkey", "company_name", "cik", "cusip", "ticker", "candidate_status"]
no_local_candidate_review = baseline_review.loc[
    baseline_review["candidate_status"].eq("No local candidate"), review_columns
].sort_values("gvkey").reset_index(drop=True)
assert len(no_local_candidate_review) == 87
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None):
    display(no_local_candidate_review)
# Reload only to verify that the in-memory audit did not alter the saved crosswalk.
pd.testing.assert_frame_equal(crosswalk_v1, pd.read_parquet(input_paths["crosswalk_v1"]))

,gvkey,company_name,cik,cusip,ticker,candidate_status
0,001177,(),(),(),(),No local candidate
1,001380,"(HESS CORP,)","(0000004447,)","(42809H107,)","(HES,)",No local candidate
2,002019,"(BANK OF NEW YORK MELLON CORP,)","(0001390777,)","(064058100,)","(BNY,)",No local candidate
3,002154,"(BEMIS CO INC,)","(0000011199,)","(081437105,)","(BMS,)",No local candidate
4,002176,"(BERKSHIRE HATHAWAY,)","(0001067983,)","(084670702,)","(BRK.B,)",No local candidate
5,002435,"(BROWN FORMAN CORP,)","(0000014693,)","(115637209,)","(BF.B,)",No local candidate
6,003231,"(COMERICA INC,)","(0000028412,)","(200340107,)","(CMA,)",No local candidate
7,003310,"(CA INC,)","(0000356028,)","(12673P105,)","(CA.1,)",No local candidate
8,004990,"(GAP INC,)","(0000039911,)","(364760108,)","(GAP,)",No local candidate
9,005667,"(HF SINCLAIR CORP,)","(0001915657,)","(403949100,)","(DINO,)",No local candidate


### Historical baseline findings — before the 2026-11-06 full-reference run

- The saved baseline has **652 unique GVKEY firms**, **539 direct candidates**, **26
  supplementary candidates** (25 CIK plus full CUSIP, one full CUSIP only), and **87
  without local candidates**. All accepted RCIDs and decision bases are null.
- The exported CIK, CUSIP8, and ticker scalars are entirely null. Notebook 01 Part 7.2
  uses a tuple-only guard; with pandas 3.0.1, string-column aggregation returns lists,
  so even singleton evidence is discarded. The diagnostic reproduces this failure
  against the current source. The historical export environment is not independently
  established, so this is a reproduced mechanism consistent with the saved artifact,
  not proof of its exact execution history.
- Company name and original full CUSIP were omitted from Notebook 01's export schema.
  Its CIK scalar was intended to be zero-normalized and CUSIP8 truncated; neither is
  a substitute for original source identifiers. This audit retains raw values and
  reconstructs evidence separately, without patching the v1 artifact.
- The same tuple-only check appears in Notebook 01 Part 5's `company_evidence` handling.
  It can empty the displayed observed-name/identifier evidence and distort identity
  review flags. Direct GVKEY availability and supplementary identifier joins use
  separate paths. Exact equality of saved and reconstructed candidate-method triples
  tests candidate preservation; it does **not** validate the saved identity-review flags.
- The 87 firms are absent only under the original three methods in the **ticker-filtered
  local extract**. This does not establish absence from the full Revelio universe.
  Ticker/name searches, historical validity, hierarchy conflicts, annual-format choices,
  and RCID adjudication were unresolved at this baseline stage. Part 2 had not yet been implemented.
  The 2026-11-06 discovery findings and current consolidation workflow appear below.
- Completeness and multiplicity diagnostics above report the actual source gaps without
  selecting preferred records. All reconstructed evidence remains in memory; no source,
  crosswalk, membership bridge, schema, or log is exported or overwritten.

## Part 2 Discovery → Consolidation → Review → Deliverables

### 2.1 Full Revelio Reference and Candidate Discovery

The reference is now present at `data/raw/revelio/eudkcvtewjnwtygx.csv.gz`.
Part 1 remains the validated historical baseline. The scan algorithm below retains
its established exact-identifier rules; historical comparisons move to Part 2.2.

#### 2.1.0 Lightweight setup and optional checkpoint restore — RUN THIS FIRST

This cell defines shared identifier helpers before discovery and loads the
consolidation functions. **It does not scan, save, or reset existing evidence.**
Leave `CHECKPOINT_TO_LOAD = None` when using the completed live kernel.
For a fresh kernel, run Part 0, set it to an existing complete version directory,
and run this cell. Restoration validates evidence checksums, small-input hashes,
large-source size/mtime, exact counts, and source links before installing state.
Do not point it at an arbitrary latest directory; select the reviewed version.

Full-reference size/mtime are not a content hash or proof of scan-time provenance.
WRDS relation, extraction settings, retrieval date, and temporal completeness remain
unverified where the original provenance fields were blank.


In [15]:
# 2.1.0 — lightweight setup; safe for the existing completed kernel
import importlib
import sys

notebook_directory = (project_root / "code/notebooks").resolve()
if str(notebook_directory) not in sys.path:
    sys.path.insert(0, str(notebook_directory))
import revelio_consolidation as matching_workflow
matching_workflow = importlib.reload(matching_workflow)
text_key = matching_workflow.text_key
positive_digits = matching_workflow.positive_digits
cusip_audit_key = matching_workflow.cusip_audit_key

RUN_FULL_REFERENCE_AUDIT = False
RUN_FULL_IDENTIFIER_SCAN = False
CHECKPOINT_TO_LOAD = (
    r"C:\Users\KPdesktop\Desktop\AI Workforce Statements"
    r"\data\clean\revelio_matching"
    r"\evidence_v1_20261109T025454049300Z"
)  # Example: project_root / "data/clean/revelio_matching/evidence_v1_YYYYMMDDTHHMMSSZ"
if CHECKPOINT_TO_LOAD is not None:
    restored_matching_state = matching_workflow.restore_checkpoint(project_root, CHECKPOINT_TO_LOAD)
    globals().update(restored_matching_state)
    print("Restored complete, validated derived evidence:", CHECKPOINT_TO_LOAD)
else:
    print("Using existing kernel evidence; no checkpoint read and no scan started.")
print("Full-reference scans disabled. Next for this iteration: 2.2.1.")


Restored complete, validated derived evidence: C:\Users\KPdesktop\Desktop\AI Workforce Statements\data\clean\revelio_matching\evidence_v1_20261109T025454049300Z
Full-reference scans disabled. Next for this iteration: 2.2.1.


#### 2.1.1 Reference inventory and provenance

**Historical preparation (2026-11-05):** the initial inventory had only the
ticker-filtered extract. **Download inspection (2026-11-06):** the gzip file had
7,634,183,551 compressed bytes and 34 columns. The saved bounded audit counted
33,961,589 parsed data rows. These observations do not verify an unrestricted WRDS
export or global RCID uniqueness. Inventory and five-row inspection below are
optional when resuming retained evidence. They do not substitute for the evidence
checkpoint and do not need to be repeated for Part 2.2.


In [16]:
from datetime import datetime, timezone
import csv
import hashlib

revelio_full_company_path = project_root / "data/raw/revelio/eudkcvtewjnwtygx.csv.gz"
revelio_full_source = None
revelio_full_audit = None
revelio_full_status = "PENDING_INSPECTION" if revelio_full_company_path.is_file() else "FILE_NOT_FOUND"
print("Checked (UTC):", datetime.now(timezone.utc).isoformat())
print("Downloaded reference (coverage unverified):", revelio_full_company_path.resolve())
display(pd.DataFrame([
    {"file": path.name, "bytes": path.stat().st_size}
    for path in sorted((project_root / "data/raw/revelio").iterdir()) if path.is_file()
]))
print("Full reference exists:", revelio_full_company_path.is_file())
if not revelio_full_company_path.is_file():
    print("FILE_NOT_FOUND: check the downloaded gzip path before inspection.")

# Fill from the actual WRDS export, not filesystem timestamps or filename guesses.
revelio_full_provenance = {
    "wrds_relation": None,
    "retrieved_utc": None,
    "query_or_download_settings": None,
    "wrds_row_count": None,
    "unrestricted_export_confirmed": False,
}

Checked (UTC): 2026-11-09T04:18:21.297064+00:00
Downloaded reference (coverage unverified): C:\Users\KPdesktop\Desktop\AI Workforce Statements\data\raw\revelio\eudkcvtewjnwtygx.csv.gz


,file,bytes
0,eudkcvtewjnwtygx.csv,25162237695
1,eudkcvtewjnwtygx.csv.gz,7634183551
2,revelio_company_mapping_20261023.csv,249090


Full reference exists: True


In [17]:
# 2.1.1a Inspection only: gzip header and first five rows. STOP after this cell.
import gzip
from itertools import islice

with gzip.open(revelio_full_company_path, "rt", encoding="utf-8-sig", newline="") as handle:
    revelio_preview_reader = csv.reader(handle)
    revelio_preview_columns = next(revelio_preview_reader)
    revelio_preview_rows = list(islice(revelio_preview_reader, 5))

# csv.reader returns strings: no numeric inference, normalization, or NA conversion.
revelio_full_preview = pd.DataFrame(
    revelio_preview_rows, columns=revelio_preview_columns, dtype="string",
)
revelio_compressed_bytes = revelio_full_company_path.stat().st_size
print("Compressed file:", revelio_full_company_path.resolve())
print(f"Compressed file size: {revelio_compressed_bytes:,} bytes "
      f"({revelio_compressed_bytes / 1024**3:.3f} GiB)")
print("Number of columns:", len(revelio_preview_columns))
print("Column names:")
display(revelio_preview_columns)
print("First 5 rows (or fewer if the file ends):")
with pd.option_context("display.max_columns", None):
    display(revelio_full_preview)
print("STOP for researcher review. Schema/sample and file size do not establish full coverage.")


Compressed file: C:\Users\KPdesktop\Desktop\AI Workforce Statements\data\raw\revelio\eudkcvtewjnwtygx.csv.gz
Compressed file size: 7,634,183,551 bytes (7.110 GiB)
Number of columns: 34
Column names:


['ultimate_parent_rcid',
 'rcid',
 'company',
 'factset_entity_id',
 'year_founded',
 'ticker',
 'exchange_name',
 'sedol',
 'isin',
 'cusip',
 'url',
 'naics_code',
 'cik',
 'lei',
 'linkedin_url',
 'child_rcid',
 'child_company',
 'child_linkedin_url',
 'ultimate_parent_company_name',
 'gvkey',
 'ein',
 'hq_street_address',
 'hq_zip_code',
 'hq_city',
 'hq_metro_area',
 'hq_state',
 'hq_country',
 'hq_region',
 'rics_k50',
 'rics_k200',
 'rics_k400',
 'phone_number',
 'slogan',
 'description']

First 5 rows (or fewer if the file ends):


,ultimate_parent_rcid,rcid,company,factset_entity_id,year_founded,ticker,exchange_name,sedol,isin,cusip,url,naics_code,cik,lei,linkedin_url,child_rcid,child_company,child_linkedin_url,ultimate_parent_company_name,gvkey,ein,hq_street_address,hq_zip_code,hq_city,hq_metro_area,hq_state,hq_country,hq_region,rics_k50,rics_k200,rics_k400,phone_number,slogan,description
0,21013947,1,"ClearSlide, Inc.",0C3V5L-E,2009,,,,,,clearslide.com,513210,,,http://linkedin.com/company/clearslide-inc-,2979896,"Crunched, Inc.",http://linkedin.com/company/crunched,BigTinCan Holdings Ltd.,,,3001 Bishop Drive,94538,San Ramon,san francisco metropolitan area,California,United States,Northern America,Information Technology Services,Enterprise Software and IT Services,Enterprise Software Solutions,,The Sales Engagement Platform Leader - Content...,ClearSlide makes it easy for sales and marketi...
1,2,2,Alzheimerfonden,0HFRW3-E,1988,,,,,,alzheimerfonden.se,813211,,549300HKKCQWIBSDLS40,http://linkedin.com/company/alzheimerfonden,2,Alzheimerfonden,http://linkedin.com/company/alzheimerfonden,Alzheimerfonden,,,Vegagatan 9,113 29,Stockholm,stockholm metropolitan area,Stockholm,Sweden,Northern Europe,Humanitarian Services,Nonprofit and Humanitarian,Humanitarian and Nonprofit,,,Alzheimerfondens fokus ligger på innovativ for...
2,3,3,Ajinga,0NYQ5C-E,2011,,,,,,ajinga.com,561312,,,http://linkedin.com/company/ajinga,3,Ajinga,http://linkedin.com/company/ajinga,Ajinga,,,,,Shanghai,shanghai metropolitan area,Shanghai,China,Eastern Asia,Human Resources Services,Employment and Staffing Services,Recruitment and Staffing Services,+862151602533,"ATS, Job Distribution, WeChat Recruiting, Refe...","AJINGA is an award-winning, China-based AI dig..."
3,4,4,"Crevasse AI Co., Ltd.",0N4W1S-E,2018,,,,,,crevasse.ai,513210,,,http://linkedin.com/company/crevasse-ai,4,"Crevasse AI Co., Ltd.",http://linkedin.com/company/crevasse-ai,"Crevasse AI Co., Ltd.",,,64 Janghan-ro,02640,Seoul,seoul metropolitan area,Seoul,South Korea,Eastern Asia,Information Technology Services,Enterprise Software and IT Services,Enterprise Software and Cloud Services,,"AI, music, streaming, recommendation, social m...",AI Startup for developing and researching on d...
4,5,5,Wav11 LLC,0PKPS0-E,2003,,,,,,wav11.com,541512,,,http://linkedin.com/company/wav11,5,Wav11 LLC,http://linkedin.com/company/wav11,Wav11 LLC,,,,,McAlester,oklahoma nonmetropolitan area,Oklahoma,United States,Northern America,Information Technology Services,Information Technology Services,IT Development and Digital Solutions,,,WAV11 takes a responsive and progressive appro...


STOP for researcher review. Schema/sample and file size do not establish full coverage.


#### 2.1.2 Acquisition history and outstanding provenance

The 2026-11-05 acquisition instructions predated the downloaded gzip. The proposed
`revelio_company_reference_full.csv` destination and decompression instructions are
historical and superseded by the actual gzip path. The complete instructions remain
in the dated notebook archive. No new download or decompression is needed here.

Confirm the actual WRDS relation, unrestricted export settings, retrieval date and
job row count from the original acquisition records when available. Neither a
filename, filesystem timestamp nor candidate coverage supplies these facts.


#### 2.1.3 Part 2.1B: gzip-aware, bounded-memory diagnostics

This first pass validates the CSV header, counts parsed data rows, and summarizes
missingness and nonblank coverage for `rcid`, `company`, `gvkey`, `cik`, `cusip`,
`ticker`, and `ultimate_parent_rcid`. It replaces the former full-file loader.
The saved UnicodeDecodeError came from decoding compressed bytes as plain UTF-8;
header inspection now uses `gzip.open`, and the chunk reader explicitly uses gzip.

Header labels must be nonempty and unique after trimming and lowercasing for lookup
only. Original labels and field values are unchanged. Missing requested columns are
reported as absent, with counts and percentages unavailable rather than fabricated.
A missing RCID or company column requires review, but does not prevent counting rows.

The initial chunk size is **10,000 rows**, with only available requested columns
materialized. If none exist, one source column is read solely to count rows. Each
chunk and its temporary Series are discarded after updating scalar counters; no
row-level table, list of chunks, or identifier universe is retained or returned.
Memory depends on chunk size and field lengths, not total row count; this is not a
fixed byte limit. Long text columns outside the requested fields are not retained.
The scan still decompresses the complete file and may take time.

All loaded values use string dtype with NA inference disabled, preserving empty
strings, leading zeros, whitespace, and literal `NA`. Empty strings and whitespace-only
strings are counted separately. Missing/blank is their sum; coverage is nonblank rows
divided by all parsed data rows. Literal `NA`, `.`, and other nonblank tokens count as
present in this pass, without asserting that they are valid identifiers. A zero-row
file has undefined percentages. Blank lines are not silently skipped; this counts
CSV records, not physical lines, and quoted embedded newlines remain within records.
Parser errors abort the scan rather than silently skipping malformed records.

**Original audit objectives retained, but pending:** identifier format validity and
possible missing-token review; missingness outside these seven fields; exact duplicates;
raw and trimmed RCID uniqueness/multiplicity; per-RCID conflicting values; parent/child
missing targets, self-links, multiple targets, onward-parent joins, and name consistency.
No global duplicate/groupby/hierarchy checks, SHA256, matching, or acceptance occur here.
Header validation and nonblank coverage do not establish the WRDS relation, unrestricted
export, unique-company count, historical validity, or complete coverage.


In [18]:
import csv
import gzip


def audit_revelio_full_reference(path, chunksize=10_000):
    """Part 2.1B only: return aggregate counts, never row-level records."""
    if isinstance(chunksize, bool) or not isinstance(chunksize, int) or chunksize <= 0:
        raise ValueError("chunksize must be a positive integer.")
    with gzip.open(path, "rt", encoding="utf-8-sig", newline="") as handle:
        header = next(csv.reader(handle, strict=True), [])
    labels = [label.strip().lower() for label in header]
    if not labels or any(not label for label in labels) or len(labels) != len(set(labels)):
        raise ValueError("Empty or ambiguous column labels; inspect the original header without rewriting it.")
    lookup = dict(zip(labels, header))
    fields = ["rcid", "company", "gvkey", "cik", "cusip", "ticker", "ultimate_parent_rcid"]
    present = [field for field in fields if field in lookup]
    selected = [lookup[field] for field in present] or [header[0]]
    counts = {field: {"empty_rows": 0, "whitespace_only_rows": 0} for field in present}
    total_rows = 0
    chunks_processed = 0

    with pd.read_csv(
        path, compression="gzip", encoding="utf-8-sig", dtype="string",
        keep_default_na=False, na_filter=False, skip_blank_lines=False,
        usecols=selected, chunksize=chunksize, on_bad_lines="error",
    ) as reader:
        for chunk in reader:
            total_rows += len(chunk)
            chunks_processed += 1
            for field in present:
                values = chunk[lookup[field]]
                empty = values.eq("")
                whitespace_only = values.str.strip().eq("") & ~empty
                counts[field]["empty_rows"] += int(empty.sum())
                counts[field]["whitespace_only_rows"] += int(whitespace_only.sum())
                del values, empty, whitespace_only
            del chunk

    records = []  # Seven aggregate records only; never source rows or chunks.
    for field in fields:
        exists = field in lookup
        empty = counts[field]["empty_rows"] if exists else pd.NA
        whitespace = counts[field]["whitespace_only_rows"] if exists else pd.NA
        missing = empty + whitespace if exists else pd.NA
        nonblank = total_rows - missing if exists else pd.NA
        records.append({
            "field": field, "column_present": exists, "total_rows": total_rows,
            "empty_rows": empty, "whitespace_only_rows": whitespace,
            "missing_or_blank_rows": missing, "nonblank_rows": nonblank,
            "missing_or_blank_pct": 100 * missing / total_rows if exists and total_rows else pd.NA,
            "nonblank_coverage_pct": 100 * nonblank / total_rows if exists and total_rows else pd.NA,
        })
    return {
        "summary": {"rows": total_rows, "columns": len(header),
                    "chunks_processed": chunks_processed, "chunk_size": chunksize},
        "columns": header,
        "missingness": pd.DataFrame(records),
    }


#### 2.1.4 Expensive bounded audit — disabled by default

The saved audit counted 33,961,589 rows. Reorganization does not require this pass.
The function above is unchanged. Set `RUN_FULL_REFERENCE_AUDIT = True` only for a
deliberately requested new audit; this invocation otherwise leaves kernel evidence
intact. Archived outputs document the prior run; this edited cell has no new output.


In [19]:
if RUN_FULL_REFERENCE_AUDIT:
    # Part 2.1B: rerun the function-definition cell above before this manual scan.
    revelio_full_source = None
    revelio_full_audit = None
    revelio_full_status = "PENDING_BOUNDED_SCAN"
    revelio_diagnostic_chunksize = 10_000
    if not revelio_full_company_path.is_file():
        revelio_full_status = "FILE_NOT_FOUND"
        print("FILE_NOT_FOUND: check the compressed export path.")
    else:
        revelio_full_status = "BOUNDED_SCAN_IN_PROGRESS"
        print(f"Scanning gzip in {revelio_diagnostic_chunksize:,}-row chunks; this may take time.", flush=True)
        try:
            revelio_full_audit = audit_revelio_full_reference(
                revelio_full_company_path, chunksize=revelio_diagnostic_chunksize,
            )
        except BaseException:
            revelio_full_status = "BOUNDED_SCAN_FAILED_OR_INTERRUPTED"
            raise
        revelio_full_status = "BOUNDED_PASS_COMPLETE_REVIEW_REQUIRED"
        print("Scan totals:", revelio_full_audit["summary"])
        with pd.option_context("display.max_columns", None):
            display(revelio_full_audit["missingness"])
        print(revelio_full_status)
        print("Coverage means nonblank rows, not valid identifiers or unique companies.")
        print("Duplicates, RCID uniqueness, hierarchy, provenance and full coverage remain unvalidated.")
        print("STOP for researcher review before additional diagnostics.")
else:
    print("SKIPPED: full-reference audit. Existing evidence retained.")


SKIPPED: full-reference audit. Existing evidence retained.


#### 2.1.5 Established findings and limits

**Saved execution, 2026-11-06 (not recomputed by this edit):** full-reference
discovery found 651/652 target firms, 780 pairs, 115 multi-RCID firms, and 2,001
source-row/target/method records from 780 matching source rows. The scan processed
33,961,589 reference rows. Method coverage overlaps: GVKEY 642 firms, CIK 610,
validated full CUSIP 639. There were 216 pairs absent from the earlier local union.
The old 2026-11-05 `PENDING_DOWNLOAD` statement is historical, not current status.

No global source deduplication, RCID uniqueness or ownership-tree validation has
been established. The retained eight fields have no Revelio effective dates;
annual Compustat dates are observations, not proof of historical match validity.
Parent and child identifiers are observations only. No accepted match follows from
these coverage counts. GVKEY 179700 is the saved unmatched target; Part 2.2 checks
this explicitly and preserves missing names.


### Part 2.1C-A Exact Identifier Candidate Discovery

**Dependency preparation:** run Part 1 and the lightweight 2.1.0 helper cell before
a deliberately requested discovery run. Neither Part 2.2 nor Part 2.3 is a
prerequisite. For the current iteration, skip these discovery cells and proceed
with retained evidence to 2.2.1. Both expensive passes default to disabled.

Use exactly the 652 GVKEYs in `crosswalk_v1` and all historical observations in
`compustat_evidence`, without a latest-record or additional date filter. Direct GVKEY
uses exact trimmed strings, as in Part 1 (no zero-padding or zero removal). CIK uses
the unchanged `positive_digits` helper: digits only, positive, leading zeros removed in keys only.
Full CUSIP uses the unchanged `cusip_audit_key` helper: uppercase/trim, valid 8-character values
or 9-character values with a valid check digit. Equality includes the entire value;
8 and 9 characters never match via a prefix. This is stricter than the earlier baseline
shape-only CUSIP rule, whose original candidates remain unchanged.

All three methods search all targets independently. Shared identifiers retain every
target and every nonblank RCID; no contradictory identifier suppresses evidence.
Raw Revelio identifiers and parent/child RCIDs are retained for later review, not used
for parent substitution or hierarchy analysis. Historical Compustat raw values, dates,
and formats are retained once in `full_id_source_evidence`; candidate evidence links
to all applicable original `compustat_row` values. A direct GVKEY hit can exist even
without an annual observation, because membership in the established target universe
is sufficient for that method.

The scan reads 10,000 rows per chunk and only eight source columns. Unmatched rows
are discarded. Evidence is capped at 50,000 target/RCID/method/source-row records and
20 million retained raw-text characters (conservative initial limits, not an exact
RAM guarantee). Exceeding either limit aborts without publishing partial results;
review the limits before authorizing another run. No source deduplication or RCID
uniqueness is assumed. `revelio_source_row` is the one-based CSV data-record ordinal,
excluding the header, not a physical line number. Only the retained candidate subset
is grouped for reporting. Discovery itself writes nothing; the optional derived checkpoint is separate in Part 2.2.

Firm, pair, evidence-record and distinct source-row counts are separate units.
Comparison with earlier candidates now occurs in Part 2.2, so those historical
sets are not discovery dependencies. The scan function and identifier matching
semantics are unchanged; only prerequisites, invocation guards, and reporting move.


In [20]:
# 2.1C-A.1 ? Prerequisites, immutable comparison sets, and historical lookup evidence
from collections import defaultdict
import csv
import gzip

full_id_required = ["crosswalk_v1", "compustat_evidence", "saved_candidates",
                    "positive_digits", "cusip_audit_key"]
full_id_missing = [name for name in full_id_required if name not in globals()]
if full_id_missing:
    raise RuntimeError("Run Part 1 and 2.1.0 first: " + ", ".join(full_id_missing))
full_id_targets = frozenset(crosswalk_v1.gvkey)
full_id_baseline_before = frozenset(saved_candidates)
full_id_crosswalk_before = crosswalk_v1.copy(deep=True)
assert len(full_id_targets) == len(crosswalk_v1) == 652 and crosswalk_v1.gvkey.is_unique
assert len({g for g, r, m in full_id_baseline_before}) == 565
assert crosswalk_v1.rcid.isna().all()

full_id_source_evidence = compustat_evidence[[
    "compustat_row", "gvkey_key", "gvkey", "conm", "cik", "cusip",
    "datadate", "fyear", "indfmt", "datafmt", "consol", "curcd",
]].copy()
assert full_id_source_evidence.compustat_row.is_unique
assert set(full_id_source_evidence.gvkey_key) <= full_id_targets
full_id_source_evidence["cik_comparison"] = full_id_source_evidence.cik.map(positive_digits)
full_id_source_evidence["cusip_comparison"] = full_id_source_evidence.cusip.map(
    lambda value: cusip_audit_key(value)[0])
full_id_lookup = {method: defaultdict(dict) for method in ["direct_gvkey", "cik", "full_cusip"]}
for full_id_g in sorted(full_id_targets):
    full_id_lookup["direct_gvkey"][full_id_g][full_id_g] = []
for full_id_row in full_id_source_evidence.itertuples(index=False):
    for full_id_method, full_id_key in [
        ("direct_gvkey", full_id_row.gvkey_key),
        ("cik", full_id_row.cik_comparison), ("full_cusip", full_id_row.cusip_comparison),
    ]:
        if full_id_key:
            full_id_lookup[full_id_method][full_id_key].setdefault(
                full_id_row.gvkey_key, []).append(int(full_id_row.compustat_row))
# Freeze row-reference lists once; candidate records share these compact tuples.
full_id_lookup = {m: {key: {g: tuple(rows) for g, rows in owners.items()}
                         for key, owners in mapping.items()}
                  for m, mapping in full_id_lookup.items()}
print("Targets:", len(full_id_targets), "| historical Compustat rows:", len(full_id_source_evidence))


Targets: 652 | historical Compustat rows: 5725


In [21]:
# 2.1C-A.2 ? Bounded full-reference scan; retains only target-matching evidence

def discover_full_identifier_candidates(path, lookup, chunksize=10_000,
                                        max_evidence=50_000, max_raw_characters=20_000_000):
    fields = ["rcid", "company", "gvkey", "cik", "cusip",
              "ultimate_parent_rcid", "child_rcid", "ticker"]
    with gzip.open(path, "rt", encoding="utf-8-sig", newline="") as handle:
        header = next(csv.reader(handle, strict=True), [])
    labels = [label.strip().lower() for label in header]
    if not labels or any(not x for x in labels) or len(set(labels)) != len(labels):
        raise ValueError("Empty or ambiguous CSV header; stop for review.")
    columns = dict(zip(labels, header))
    missing = set(fields) - set(columns)
    if missing:
        raise ValueError("Required evidence columns absent: " + ", ".join(sorted(missing)))
    evidence = []
    rows_scanned = raw_characters = blank_rcid_hits = 0
    output_columns = ["gvkey", "rcid", "method", "comparison_key", "revelio_source_row",
                      "compustat_rows"] + ["revelio_" + f + "_raw" for f in fields]
    with pd.read_csv(path, compression="gzip", encoding="utf-8-sig", dtype="string",
                     keep_default_na=False, na_filter=False, skip_blank_lines=False,
                     usecols=[columns[f] for f in fields], chunksize=chunksize,
                     on_bad_lines="error") as reader:
        for chunk in reader:
            chunk = chunk.rename(columns={columns[f]: f for f in fields}).reset_index(drop=True)
            keys = pd.DataFrame(index=chunk.index)
            keys["direct_gvkey"] = chunk.gvkey.str.strip()
            cik = chunk.cik.str.strip()
            keys["cik"] = cik.str.lstrip("0").where(cik.str.fullmatch(r"[0-9]+"), "")
            # Lookup keys already passed the existing full-CUSIP validator.
            # Exact equality to those keys also validates the matching reference value.
            keys["full_cusip"] = chunk.cusip.str.strip().str.upper()
            hit = pd.Series(False, index=chunk.index)
            for method in lookup:
                hit |= keys[method].isin(lookup[method])
            for position in chunk.index[hit]:
                raw = tuple(chunk.at[position, f] for f in fields)
                rcid = raw[0].strip()
                if not rcid:
                    blank_rcid_hits += 1
                    continue  # Count explicitly; a blank RCID cannot define a candidate pair.
                for method in lookup:
                    key = keys.at[position, method]
                    for gvkey, source_rows in lookup[method].get(key, {}).items():
                        next_characters = raw_characters + sum(len(value) for value in raw)
                        if len(evidence) >= max_evidence or next_characters > max_raw_characters:
                            raise RuntimeError("Candidate retention limit reached; no complete results published. "
                                               "Review accumulation before changing limits and rescanning.")
                        evidence.append((gvkey, rcid, method, key,
                                         rows_scanned + int(position) + 1, source_rows, *raw))
                        raw_characters = next_characters
            rows_scanned += len(chunk)
            del chunk, keys, cik, hit
    return pd.DataFrame.from_records(evidence, columns=output_columns), {
        "rows_scanned": rows_scanned, "identifier_hit_rows_without_rcid": blank_rcid_hits,
        "retained_evidence_records": len(evidence), "retained_raw_characters": raw_characters,
    }

if RUN_FULL_IDENTIFIER_SCAN:
    full_id_evidence = full_id_scan_summary = full_id_pairs = None
    full_id_status = "SCAN_IN_PROGRESS"
    print("Scanning all gzip records in 10,000-row chunks; no intermediate results are final.", flush=True)
    try:
        full_id_evidence, full_id_scan_summary = discover_full_identifier_candidates(
            revelio_full_company_path, full_id_lookup)
        if full_id_scan_summary["rows_scanned"] != 33_961_589:
            raise ValueError("Row count differs from the established Part 2.1B count; investigate before reporting candidates.")
    except BaseException:
        full_id_evidence = full_id_scan_summary = None
        full_id_status = "FAILED_OR_INTERRUPTED"
        raise
    full_id_status = "SCAN_COMPLETE_REVIEW_REQUIRED"
else:
    print("SKIPPED: full-reference discovery. Retained full_id_evidence was not reset.")


SKIPPED: full-reference discovery. Retained full_id_evidence was not reset.


In [22]:
# 2.1C-A.3 — lightweight reporting; historical-set reconciliation is now in 2.2
if globals().get("full_id_status") != "SCAN_COMPLETE_REVIEW_REQUIRED":
    raise RuntimeError("Restore complete retained evidence; no scan is started by this report.")
full_id_pairs = full_id_evidence.groupby(["gvkey", "rcid"], sort=True).agg(
    methods=("method", lambda values: tuple(sorted(set(values)))),
    source_rows=("revelio_source_row", "nunique"), evidence_records=("method", "size"),
).reset_index()
assert len(full_id_pairs) == 780 and full_id_pairs.gvkey.nunique() == 651
assert len(full_id_evidence) == 2001
full_id_method_summary = full_id_evidence.groupby("method").agg(
    target_firms=("gvkey", "nunique"), reference_source_rows=("revelio_source_row", "nunique"),
    evidence_records=("method", "size"),
)
display(full_id_method_summary)
print("Full-reference candidate pairs:", len(full_id_pairs))
print("Firms with multiple full-reference RCIDs:", int(full_id_pairs.groupby("gvkey").rcid.nunique().gt(1).sum()))
print("Unmatched GVKEYs:", sorted(set(crosswalk_v1.gvkey) - set(full_id_pairs.gvkey)))
print("Candidate availability only. Continue to 2.2; no identities accepted.")


,target_firms,reference_source_rows,evidence_records
method,,,
cik,610,720,720
direct_gvkey,642,642,642
full_cusip,639,639,639


Full-reference candidate pairs: 780
Firms with multiple full-reference RCIDs: 115
Unmatched GVKEYs: ['179700']
Candidate availability only. Continue to 2.2; no identities accepted.


### 2.2 Candidate Consolidation and Multi-method Evidence Integration

This section imports the completed evidence; it does not rerun ticker/name searches
or open the full reference. Distinct GVKEY–RCID pairs define the candidate table.
The evidence ledger retains each original full-reference source-row/target/method
record, every earlier supplementary evidence record, and reconstructed Notebook 01
source-row/method support under its original rules. Source-row ordinals are
one-based parsed CSV data records, excluding headers. Full and filtered Revelio
row references occupy separate namespaces.

Notebook 01 shape-only full-CUSIP evidence remains present even if it fails the
later check-digit validator. Validated full-CUSIP support requires whole-string
equality: 8 and 9 characters remain different discovery keys. Legacy CUSIP flags
are retained inside `legacy_flags` and do not become entity-conflict judgments.
Dates are carried only where observed. No date is invented for the Revelio snapshot.

Expected reconciliation from the saved run: 565 original pairs + 73 supplementary
pairs = 638 earlier pairs; 564 intersect the full set; 216 are full-only; 74 are
earlier-only; union = 854. Assertions require these observed relationships rather
than silently dropping candidates to reproduce 780.

#### 2.2.1 Verify retained inputs — RUN after 2.1.0

Requires the original live kernel or an explicitly restored checkpoint. Missing
inputs stop here with their names. Displayed notebook outputs cannot replace the
full 2,001-row evidence object. This cell does not write anything.


In [23]:
# 2.2.1 — verify retained inputs, without rerunning any discovery
part22_input_state = matching_workflow.require_state(globals())
part22_input_checks = matching_workflow.validate_retained_state(part22_input_state)
part22_input_digest = matching_workflow.state_digest(part22_input_state)
display(part22_input_checks)
print("Input evidence digest:", part22_input_digest)
print("Retained state verified; no candidate accepted and no files written.")


,check,observed_value,status
0,target_firms,652,PASS
1,original_pairs,565,PASS
2,original_method_triples,1638,PASS
3,earlier_supplement_pairs,73,PASS
4,full_reference_pairs,780,PASS
5,full_reference_evidence_records,2001,PASS
6,full_reference_candidate_firms,651,PASS
7,full_reference_multiple_rcid_firms,115,PASS
8,accepted_rcids,0,PASS


Input evidence digest: d784f3432b3a86502c797104ba7e9c8df9760bb7c6edaddbf5fac8f72d7786dd
Retained state verified; no candidate accepted and no files written.


#### 2.2.2 Optional preservation checkpoint — explicitly controlled write

**Recommended before restarting the kernel.** Set `WRITE_EVIDENCE_CHECKPOINT = True`
in this cell only if you want to save the small retained evidence now, then run it.
This is a restart checkpoint, not the professor-facing export. It writes a new
UTC-versioned directory under `data/clean/revelio_matching/`, containing typed JSON
and a manifest. Nested raw strings, tuples, nulls, source rows and dataframe dtypes
round-trip before the completion manifest is written. Existing directories are
never overwritten. An interrupted directory without a manifest is not loadable.

Small source files are hashed; the 7.6-GB gzip and CRSP file are only stat'ed.
These fingerprints are recorded now, not claimed as scan-time hashes. No raw
reference rows beyond the retained candidate subset are exported. Copy the printed
path to `CHECKPOINT_TO_LOAD` for a later fresh-kernel restart. Leave False to keep
this entire execution chunk read-only.


In [24]:
# 2.2.2 — optional derived evidence preservation; no automatic write
WRITE_EVIDENCE_CHECKPOINT = False
if WRITE_EVIDENCE_CHECKPOINT:
    from datetime import datetime, timezone
    checkpoint_version = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    evidence_checkpoint_path = project_root / "data/clean/revelio_matching" / ("evidence_v1_" + checkpoint_version)
    matching_workflow.capture_checkpoint(globals(), project_root, evidence_checkpoint_path)
    print("CHECKPOINT COMPLETE:", evidence_checkpoint_path.resolve())
else:
    print("Checkpoint write disabled. Preserve this kernel until you choose to save it.")


Checkpoint write disabled. Preserve this kernel until you choose to save it.


#### 2.2.3 Assemble the complete candidate universe — in memory only

`consolidated_evidence` retains source-supported methods, raw values and row links.
`consolidated_candidates` has exactly one row per distinct pair, with all names,
identifiers, method support, dates and observed hierarchy. `has_identifier_conflict`
is a **review flag** for a nonmissing exact-GVKEY or normalized-CIK disagreement,
including mixed evidence. It is not a false-match verdict and does not use CUSIP
variation. Part 2.3 will contextualize these diagnostics across sources and dates.

`consolidated_firm_index` preserves all 652 target GVKEYs and their ordering.
Multiple Compustat names are displayed as a labeled collection, not a preferred
legal name; missing names remain null. All RCIDs remain visible and unaccepted.
No leading RCID or quality category is assigned in this section.


In [25]:
# 2.2.3 — consolidate; no source reads, discovery, export, or acceptance
part22_results = matching_workflow.consolidate(part22_input_state)
globals().update(part22_results)
display(candidate_reconciliation)
display(consolidation_method_summary)
print("Candidate-covered firms:", int(consolidated_firm_index.candidate_count.gt(0).sum()), "/ 652")
print("Consolidated candidate pairs:", len(consolidated_candidates))
print("Firms with multiple consolidated RCIDs:", int(consolidated_firm_index.multiple_rcid_flag.sum()))
print("Evidence records by origin (different source grains; not independent votes):")
display(consolidated_evidence.groupby("origin").size().rename("evidence_records").to_frame())


,candidate_pair_set,candidate_pairs
0,Notebook 01,565
1,Earlier supplement,73
2,Earlier union,638
3,Full reference,780
4,Earlier/full intersection,564
5,Full only,216
6,Earlier only — retained,74
7,Consolidated union,854


,origin,method,target_firms,evidence_records,candidate_pairs
0,earlier_local,ticker,43,276,73
1,full_reference,cik,610,720,720
2,full_reference,direct_gvkey,642,642,642
3,full_reference,full_cusip,639,639,639
4,notebook01,cik,536,536,536
5,notebook01,direct_gvkey,539,539,539
6,notebook01,full_cusip,563,563,563


Candidate-covered firms: 651 / 652
Consolidated candidate pairs: 854
Firms with multiple consolidated RCIDs: 151
Evidence records by origin (different source grains; not independent votes):


,evidence_records
origin,
earlier_local,276
full_reference,2001
notebook01,1638


#### 2.2.4 Review previews and unresolved target

The membership table makes earlier-only alternatives explicit. Each pair's
`evidence_ids` resolves to the ledger; `compustat_rows` identifies all annual rows
used for diagnostic comparison, while `supporting_compustat_rows` identifies
rows linked to discovery evidence. Historical CRSP row/date/link details remain
in the original supplementary `source_evidence`, including P/C overlap metadata.
Revelio parent and child IDs are retained without assigning relationships or
substituting an RCID. Previews are bounded; full tables remain in memory.


In [28]:
# 2.2.4 — bounded previews; these are not final professor-facing tables
preview_columns = ["gvkey", "rcid", "research_company_names", "revelio_company_names",
                   "evidence_methods", "evidence_origins", "validated_full_cusip_support",
                   "has_identifier_conflict", "candidate_count", "rcid_target_count"]
with pd.option_context("display.max_columns", None, "display.max_colwidth", 90):
    display(consolidated_candidates[preview_columns].head(12))
    display(consolidated_firm_index.head(12))
    display(consolidated_unmatched)
earlier_only_pair_keys = candidate_membership.loc[~candidate_membership.in_full_reference, ["gvkey", "rcid"]]
earlier_only_candidate_preview = earlier_only_pair_keys.merge(
    consolidated_candidates[preview_columns], on=["gvkey", "rcid"], validate="one_to_one")
display(earlier_only_candidate_preview.head(12))
display(candidate_membership.groupby(["in_notebook01", "in_earlier_supplement", "in_full_reference"])
        .size().rename("candidate_pairs").to_frame())


,gvkey,rcid,research_company_names,revelio_company_names,evidence_methods,evidence_origins,validated_full_cusip_support,has_identifier_conflict,candidate_count,rcid_target_count
0,001045,21007446,"(AMERICAN AIRLINES GROUP INC,)","(American Airlines Group, Inc.,)","(cik, direct_gvkey, full_cusip)","(full_reference, notebook01)",True,False,2,1
1,001045,589078,"(AMERICAN AIRLINES GROUP INC,)","(AMR Corp.,)","(cik,)","(full_reference,)",False,False,2,1
2,001075,368391,"(PINNACLE WEST CAPITAL CORP,)","(Pinnacle West Capital Corp.,)","(cik, direct_gvkey, full_cusip)","(full_reference, notebook01)",True,False,1,1
3,001078,22141982,"(ABBOTT LABORATORIES,)","(Abbott Laboratories,)","(cik, direct_gvkey, full_cusip)","(full_reference, notebook01)",True,False,1,1
4,001161,1292281,"(ADVANCED MICRO DEVICES,)","(Advanced Micro Devices, Inc.,)","(cik, direct_gvkey, full_cusip)","(full_reference, notebook01)",True,False,1,1
5,001177,1328385,(),"(Aetna, Inc.,)","(direct_gvkey,)","(full_reference,)",False,False,2,1
6,001177,254524,(),"(Afentra Plc,)","(ticker,)","(earlier_local,)",False,False,2,1
7,001209,1511315,"(AIR PRODUCTS & CHEMICALS INC,)","(Air Products & Chemicals, Inc.,)","(cik, direct_gvkey, full_cusip)","(full_reference, notebook01)",True,False,1,1
8,001230,22142001,"(ALASKA AIR GROUP INC,)","(Alaska Air Group, Inc.,)","(cik, direct_gvkey, full_cusip)","(full_reference, notebook01)",True,False,1,1
9,001300,20924380,"(HONEYWELL INTERNATIONAL INC,)","(Honeywell International, Inc.,)","(cik, direct_gvkey, full_cusip)","(full_reference, notebook01)",True,False,2,1


,gvkey,research_company_names,research_company_name,candidate_rcids,candidate_count,multiple_rcid_flag,accepted_rcid,human_review_status
0,001045,"(AMERICAN AIRLINES GROUP INC,)",AMERICAN AIRLINES GROUP INC,"(21007446, 589078)",2,True,<NA>,not_reviewed
1,001075,"(PINNACLE WEST CAPITAL CORP,)",PINNACLE WEST CAPITAL CORP,"(368391,)",1,False,<NA>,not_reviewed
2,001078,"(ABBOTT LABORATORIES,)",ABBOTT LABORATORIES,"(22141982,)",1,False,<NA>,not_reviewed
3,001161,"(ADVANCED MICRO DEVICES,)",ADVANCED MICRO DEVICES,"(1292281,)",1,False,<NA>,not_reviewed
4,001177,(),<NA>,"(1328385, 254524)",2,True,<NA>,not_reviewed
5,001209,"(AIR PRODUCTS & CHEMICALS INC,)",AIR PRODUCTS & CHEMICALS INC,"(1511315,)",1,False,<NA>,not_reviewed
6,001230,"(ALASKA AIR GROUP INC,)",ALASKA AIR GROUP INC,"(22142001,)",1,False,<NA>,not_reviewed
7,001300,"(HONEYWELL INTERNATIONAL INC,)",HONEYWELL INTERNATIONAL INC,"(20924380, 22142013)",2,True,<NA>,not_reviewed
8,001327,"(SKYWORKS SOLUTIONS INC,)",SKYWORKS SOLUTIONS INC,"(47589,)",1,False,<NA>,not_reviewed
9,001380,"(HESS CORP,)",HESS CORP,"(22593742, 902387)",2,True,<NA>,not_reviewed


,gvkey,research_company_names,research_company_name,candidate_rcids,candidate_count,multiple_rcid_flag,accepted_rcid,human_review_status
625,179700,(),<NA>,(),0,False,<NA>,not_reviewed


,gvkey,rcid,research_company_names,revelio_company_names,evidence_methods,evidence_origins,validated_full_cusip_support,has_identifier_conflict,candidate_count,rcid_target_count
0,001177,254524,(),"(Afentra Plc,)","(ticker,)","(earlier_local,)",False,False,2,1
1,001380,22593742,"(HESS CORP,)","(Hydraulic Elements & Systems AD,)","(ticker,)","(earlier_local,)",False,True,2,1
2,002154,103798355,"(BEMIS CO INC,)","(Baominh Securities Co.,)","(ticker,)","(earlier_local,)",False,False,4,1
3,002154,217756,"(BEMIS CO INC,)","(Braemar Plc,)","(ticker,)","(earlier_local,)",False,True,4,1
4,002176,1446623,"(BERKSHIRE HATHAWAY,)","(SSIF BRK Financial Group SA,)","(ticker,)","(earlier_local,)",False,False,5,1
5,002176,20948983,"(BERKSHIRE HATHAWAY,)","(Brookside Energy Ltd.,)","(ticker,)","(earlier_local,)",False,True,5,1
6,002176,292851,"(BERKSHIRE HATHAWAY,)","(Brooks MacDonald Group Plc,)","(ticker,)","(earlier_local,)",False,True,5,1
7,002435,1098506,"(BROWN FORMAN CORP,)","(Bolognafiere SPA,)","(ticker,)","(earlier_local,)",False,True,2,1
8,003231,104063379,"(COMERICA INC,)","(Carma Ltd.,)","(ticker,)","(earlier_local,)",False,False,3,1
9,003231,802826,"(COMERICA INC,)","(Ciments du Maroc SA,)","(ticker,)","(earlier_local,)",False,True,3,1


candidate_pairs
in_notebook01 in_earlier_supplement in_full_reference                 
False         False                 True                           216
              True                  False                           73
True          False                 False                            1
                                    True                           564

#### 2.2.5 Preservation checks and execution checkpoint — STOP HERE

Send the input checks, reconciliation table, method summary, printed firm/pair/
multi-RCID counts, source-membership counts, unmatched row, and final PASS output.
Also send the checkpoint path if you enabled preservation, or the complete traceback
if any assertion fails. No Part 2.3 or final export should be run yet.

The candidate union is an upper bound on candidate availability. Keeping a weak
earlier-only candidate is evidence preservation, not endorsement. A zero-candidate
firm remains in the 652-row index. Accepted-match coverage remains zero.


In [29]:
# 2.2.5 — first execution checkpoint; stop after this cell
assert matching_workflow.state_digest(part22_input_state) == part22_input_digest
pd.testing.assert_frame_equal(crosswalk_v1, part22_input_state["crosswalk_v1"])
assert set(consolidated_firm_index.gvkey) == set(crosswalk_v1.gvkey)
assert consolidated_firm_index.accepted_rcid.isna().all()
assert consolidated_firm_index.human_review_status.eq("not_reviewed").all()
display(consolidation_checks)
print("PASS: Part 2.2 reconciliation and input preservation checks.")
print("Accepted matches: 0 / 652. These are candidate tables, not validated final identities.")
print("STOP. Send the displayed checks and summaries before Part 2.3 is implemented.")


,check,observed_value,status
0,target_firms,652,PASS
1,original_pairs,565,PASS
2,original_method_triples,1638,PASS
3,earlier_supplement_pairs,73,PASS
4,full_reference_pairs,780,PASS
5,full_reference_evidence_records,2001,PASS
6,full_reference_candidate_firms,651,PASS
7,full_reference_multiple_rcid_firms,115,PASS
8,accepted_rcids,0,PASS
9,consolidated_pairs,854,PASS


PASS: Part 2.2 reconciliation and input preservation checks.
Accepted matches: 0 / 652. These are candidate tables, not validated final identities.
STOP. Send the displayed checks and summaries before Part 2.3 is implemented.


### 2.3 Candidate Quality, Ambiguity and Match Review

**Scope and execution plan (01b only).** Part 2.2 is the validated baseline.
Run the seven new code cells **2.3.1 through 2.3.7**, in order, then stop.
They are self-contained on a fresh kernel; do not Run All or rerun discovery.
This checkpoint builds provisional review tables **in memory only**. No exports,
source changes, checkpoint writes, remote uploads, or acceptance decisions occur.

1. Restore the explicitly named checkpoint with the existing checksum/provenance
   workflow, and reconstruct the unchanged Part 2.2 consolidation in a separate namespace.
2. Assess identifier agreement and data gaps against the retained evidence ledger.
3. Build candidate, firm, and overlapping review-priority tables; run synthetic
   rule checks and preservation assertions.
4. After the researcher returns the checkpoint results, resolve issues and prepare
   CSV/Parquet, the evidence ledger, definitions/provenance, and meeting Markdown
   in a **new, exclusively created version directory**. Exports are deferred to that
   next iteration, as requested.

**Evidence rules, not identity probabilities.** Raw strings and all alternatives
are retained. Comparison-only CIK normalization removes leading zero padding;
GVKEY comparison follows Part 2.2's exact trimmed rule. Full CUSIP comparisons
retain all 8 or 9 characters and require the existing validity checks. An 8/9
prefix overlap is a separate format diagnostic, never a new discovery match.
A differing CUSIP can reflect security/format/time differences and is not by itself
an entity conflict. No merger, ownership history, or identity transition is inferred.

Category precedence:
- **Conflicting evidence:** observed GVKEY or normalized CIK disjoint/mixed evidence,
  or an RCID shared by target GVKEYs. This is a research conflict, not proof of error.
- **Insufficient evidence:** no linked ledger records.
- **Ambiguous multiple-RCID candidate:** more than one candidate for the GVKEY;
  every alternative remains visible, regardless of identifier strength.
- **Insufficient evidence:** no validated exact identifier type (ticker/name alone).
- **Strongly corroborated candidate:** a single candidate, at least two validated
  identifier types co-occurring on one retained full-reference row, no identifier
  conflict, CUSIP discrepancy/format-only overlap, invalid identifier values,
  observed different parent/child RCID, target identifier variation, missing dated
  study-window evidence, or legacy name/ticker/link concern.
- **Plausible candidate requiring review:** remaining exact-identifier candidates.
- **Unmatched:** firm-level only, no candidate.

A **provisional leader** is populated only for a singleton strongly corroborated
candidate. Multiple-RCID firms remain unranked. No candidate is accepted:
`accepted_rcid` is null and `human_review_status` is `unreviewed`.
The firm category prioritizes any conflict, then multiple RCIDs, then its singleton
category. Separate flags retain overlapping concerns hidden by category precedence.

**Research validity, 2018–2025.** Matching methods and repeated source rows are not
independent external verification. Both Revelio extracts belong to one provider;
external verification has not been performed. Even two identifiers on one reference
row only corroborate the snapshot candidate. Compustat/ticker observation dates
describe the source evidence, not the effective dates of the RCID relationship.
Every firm and candidate retains an unverified historical-validity status.
Observed hierarchy differences and identifier variation trigger research questions,
not inferred corporate events. Lack of a detected conflict is not proof of correctness.


In [1]:
# 2.3.1 — restore the validated checkpoint; no full-reference scan
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

for part23_root in (Path("."), Path(".."), Path("../..")):
    if (part23_root / "data/raw").is_dir():
        part23_root = part23_root.resolve()
        break
else:
    raise FileNotFoundError("Run from the project root, code/, or code/notebooks/.")
part23_module_dir = str(part23_root / "code/notebooks")
if part23_module_dir not in sys.path:
    sys.path.insert(0, part23_module_dir)
import revelio_consolidation as matching_workflow

part23_checkpoint = part23_root / (
    "data/clean/revelio_matching/evidence_v1_20261109T025454049300Z"
)
part23_state = matching_workflow.restore_checkpoint(part23_root, part23_checkpoint)
part23_state_digest = matching_workflow.state_digest(part23_state)
part23_consolidation = matching_workflow.consolidate(part23_state)
part23_inputs = {
    "candidates": part23_consolidation["consolidated_candidates"],
    "evidence": part23_consolidation["consolidated_evidence"],
    "firms": part23_consolidation["consolidated_firm_index"],
    "membership": part23_consolidation["candidate_membership"],
}
part23_inputs_digest = matching_workflow.state_digest(part23_inputs)
# Existing Part 2.2 objects, if present, must agree; never replace them.
for part23_old_name, part23_key in (
    ("consolidated_candidates", "candidates"), ("consolidated_evidence", "evidence"),
    ("consolidated_firm_index", "firms"), ("consolidated_firms", "firms"),
):
    if part23_old_name in globals():
        pd.testing.assert_frame_equal(globals()[part23_old_name], part23_inputs[part23_key])
assert part23_inputs["firms"].accepted_rcid.isna().all()
print("Restored:", part23_checkpoint)
print("PASS: checkpoint provenance/checksum and unchanged Part 2.2 consolidation.")
print("Input digest:", part23_inputs_digest)
print("No reference scan, export, or acceptance performed.")


Restored: C:\Users\KPdesktop\Desktop\AI Workforce Statements\data\clean\revelio_matching\evidence_v1_20261109T025454049300Z
PASS: checkpoint provenance/checksum and unchanged Part 2.2 consolidation.
Input digest: 9651374a13598e2817a56647671c1c6e3c1df71e27d9613013ba1b37361cae95
No reference scan, export, or acceptance performed.


#### 2.3.2 Comparison rules

Definitions only. Review the category precedence above before running the assessment.


In [2]:
# 2.3.2 — comparison and category rules (definitions only)
PART23_RULE_VERSION = "revelio-review-v1-2026-11-08"
PART23_STUDY_YEARS = tuple(range(2018, 2026))
PART23_CONFLICT_STATES = {"disagreement_all_observed", "mixed_agreement_and_disagreement"}

def part23_cusip_key(value):
    return matching_workflow.cusip_audit_key(value)[0]

def part23_compare(left, right, normalizer):
    # Original values remain in the input columns; these keys are comparison-only.
    left_raw = tuple(v for v in left if matching_workflow.text_key(v))
    right_raw = tuple(v for v in right if matching_workflow.text_key(v))
    a = {normalizer(v) for v in left_raw} - {""}
    b = {normalizer(v) for v in right_raw} - {""}
    invalid = tuple(sorted({str(v) for v in left_raw + right_raw if not normalizer(v)}))
    status = ("missing_or_invalid" if not a or not b else
              "disagreement_all_observed" if not a & b else
              "agreement" if b <= a else "mixed_agreement_and_disagreement")
    return status, tuple(sorted(a)), tuple(sorted(b)), invalid

def part23_exact_methods(group):
    return tuple(sorted({
        row.method for row in group.itertuples(index=False)
        if row.method in {"direct_gvkey", "cik"}
        or (row.method == "full_cusip" and row.validated_full_cusip_support)
    }))

def part23_category(row):
    if row["identifier_conflict_flag"] or row["rcid_shared_across_targets"]:
        return "Conflicting evidence"
    if not row["evidence_record_count"]:
        return "Insufficient evidence"
    if row["multiple_rcid_flag"]:
        return "Ambiguous multiple-RCID candidate"
    if not row["supporting_identifier_count"]:
        return "Insufficient evidence"
    if row["strong_snapshot_corroboration"]:
        return "Strongly corroborated candidate"
    return "Plausible candidate requiring review"

def part23_followup(reasons):
    actions = ["Verify entity scope and dated RCID relationship for 2018–2025 using external sources"]
    if "unmatched" in reasons:
        actions.append("Research missing target identifiers and candidate absence")
    if "multiple_rcids" in reasons:
        actions.append("Compare every alternative RCID; do not select from method counts alone")
    if "shared_rcid" in reasons or "identifier_conflict" in reasons:
        actions.append("Reconcile shared RCID and/or contradictory GVKEY/CIK against cited source rows")
    if "historical_only" in reasons:
        actions.append("Investigate absence from full-reference matches; verify historical ticker/identifier link")
    if "historical_identifier_variation" in reasons:
        actions.append("Research dated identifier variation; do not infer an identity transition")
    if "hierarchy_scope" in reasons:
        actions.append("Verify parent/subsidiary scope and effective dates without inferring ownership")
    if "cusip_discrepancy" in reasons or "cusip_format_overlap" in reasons:
        actions.append("Check security, CUSIP format, and observation dates")
    if set(reasons) & {"insufficient_identifiers", "missing_identifier_evidence", "invalid_identifier_values",
                       "no_dated_window_evidence", "legacy_name_ticker_link_concern"}:
        actions.append("Resolve missing/invalid evidence and legacy concerns before an identity decision")
    return "; ".join(actions)


#### 2.3.3 Candidate evidence assessment

Definitions only. Counts represent distinct identifier types, not independent votes.
Raw values remain beside comparison keys. Different source dates/identifiers and
hierarchy fields flag questions; they do not establish historical transitions.


In [3]:
# 2.3.3 — candidate assessment (definition only; retains every original column)
def part23_assess_candidates(candidates, evidence, annual):
    groups = {(g, r): f for (g, r), f in evidence.groupby(["gvkey", "rcid"], sort=False)}
    assert annual.compustat_row.is_unique, "Duplicate Compustat row references"
    annual_by_row = annual.set_index("compustat_row")
    rows = []
    for original in candidates.to_dict("records"):
        row = dict(original)
        group = groups.get((row["gvkey"], row["rcid"]), evidence.iloc[:0])
        comparisons = {
            "gvkey": part23_compare((row["gvkey"],), row["revelio_gvkey_raw"], matching_workflow.text_key),
            "cik": part23_compare(row["compustat_cik_raw"], row["revelio_cik_raw"], matching_workflow.positive_digits),
            "cusip": part23_compare(row["compustat_cusip_raw"], row["revelio_cusip_raw"], part23_cusip_key),
        }
        for name, (status, target_keys, reference_keys, invalid) in comparisons.items():
            row[f"review_{name}_comparison"] = status
            row[f"{name}_target_comparison_keys"] = target_keys
            row[f"{name}_reference_comparison_keys"] = reference_keys
            row[f"{name}_invalid_observed_values"] = invalid
        methods = part23_exact_methods(group)
        full = group.loc[group.origin.eq("full_reference")]
        row["supporting_identifier_types"] = methods
        row["supporting_identifier_count"] = len(methods)
        row["full_reference_identifier_types"] = part23_exact_methods(full)
        row["max_identifiers_on_one_full_reference_row"] = max(
            (len(part23_exact_methods(f)) for _, f in full.groupby("revelio_source_row")), default=0
        )
        row["historical_only"] = full.empty
        row["identifier_conflict_flag"] = any(comparisons[k][0] in PART23_CONFLICT_STATES for k in ("gvkey", "cik"))
        row["cusip_discrepancy_flag"] = comparisons["cusip"][0] in PART23_CONFLICT_STATES
        a, b = set(comparisons["cusip"][1]), set(comparisons["cusip"][2])
        row["cusip_8_9_format_overlap"] = any(
            len(x) != len(y) and x[:8] == y[:8] for x in a for y in b
        )
        row["historical_identifier_variation_flag"] = any(len(comparisons[k][1]) > 1 for k in ("cik", "cusip"))
        row["missing_identifier_types"] = tuple(k for k, v in comparisons.items() if v[0] == "missing_or_invalid")
        row["identifier_data_quality_issue"] = any(v[3] for v in comparisons.values())
        # A child field equal to the candidate itself is not evidence of a different child.
        row["different_parent_rcids"] = tuple(v for v in row["revelio_ultimate_parent_rcid_raw"]
                                               if matching_workflow.text_key(v) not in ("", row["rcid"]))
        row["different_child_rcids"] = tuple(v for v in row["revelio_child_rcid_raw"]
                                              if matching_workflow.text_key(v) not in ("", row["rcid"]))
        row["hierarchy_scope_review"] = bool(row["different_parent_rcids"] or row["different_child_rcids"])
        legacy_flags = tuple(sorted({
            k for flags in group.legacy_flags for k, v in flags.items()
            if isinstance(v, bool) and v
        }))
        row["legacy_true_flags"] = legacy_flags
        row["legacy_name_ticker_link_concern"] = bool(set(legacy_flags) & {
            "name_disagreement", "ticker_reuse_or_conflict", "ls_link_review", "parent_scope_review"
        })
        linked_rows = sorted({int(i) for values in group.compustat_rows for i in values})
        if set(linked_rows) - set(annual_by_row.index):
            raise AssertionError("Unresolved Compustat evidence row")
        dated = pd.to_datetime(annual_by_row.loc[linked_rows, "datadate"], errors="coerce")
        years = tuple(sorted(set(dated.dropna().dt.year) & set(PART23_STUDY_YEARS)))
        row["dated_compustat_evidence_years"] = years
        row["years_without_dated_compustat_evidence"] = tuple(y for y in PART23_STUDY_YEARS if y not in years)
        legacy_dates = []
        for source in group.source_evidence:
            observed = source.get("observed_dates")
            if isinstance(observed, (list, tuple)):
                legacy_dates.extend(str(v) for v in observed if matching_workflow.text_key(v))
            if matching_workflow.text_key(source.get("datadate", "")):
                legacy_dates.append(str(source["datadate"]))
        row["legacy_source_observed_dates"] = tuple(sorted(set(legacy_dates)))
        legacy_years = set(pd.to_datetime(pd.Series(legacy_dates, dtype="string"),
                                        errors="coerce", format="mixed").dropna().dt.year)
        row["dated_source_evidence_years"] = tuple(sorted(set(years) | (legacy_years & set(PART23_STUDY_YEARS))))
        row["no_dated_window_evidence"] = not row["dated_source_evidence_years"]
        row["historical_validity_status"] = "unverified_snapshot_association"
        row["historical_identity_research_required"] = True
        row["independent_external_verification_status"] = "not_performed"
        row["strong_snapshot_corroboration"] = bool(
            row["max_identifiers_on_one_full_reference_row"] >= 2
            and not any((row["identifier_conflict_flag"], row["rcid_shared_across_targets"],
                         row["cusip_discrepancy_flag"], row["cusip_8_9_format_overlap"],
                         row["identifier_data_quality_issue"], row["hierarchy_scope_review"],
                         row["legacy_name_ticker_link_concern"], row["historical_identifier_variation_flag"],
                         row["no_dated_window_evidence"]))
        )
        flags = {
            "identifier_conflict": row["identifier_conflict_flag"],
            "shared_rcid": row["rcid_shared_across_targets"],
            "multiple_rcids": row["multiple_rcid_flag"],
            "historical_only": row["historical_only"],
            "hierarchy_scope": row["hierarchy_scope_review"],
            "historical_identifier_variation": row["historical_identifier_variation_flag"],
            "cusip_discrepancy": row["cusip_discrepancy_flag"],
            "cusip_format_overlap": row["cusip_8_9_format_overlap"],
            "missing_identifier_evidence": bool(row["missing_identifier_types"]),
            "invalid_identifier_values": row["identifier_data_quality_issue"],
            "insufficient_identifiers": not methods or group.empty,
            "no_dated_window_evidence": row["no_dated_window_evidence"],
            "legacy_name_ticker_link_concern": row["legacy_name_ticker_link_concern"],
        }
        row["review_reasons"] = tuple(k for k, enabled in flags.items() if enabled)
        row["review_category"] = part23_category(row)
        row["rule_explanation"] = (
            f"{row['review_category']}; exact identifier types={methods}; "
            f"maximum co-occurring full-reference types={row['max_identifiers_on_one_full_reference_row']}; "
            f"review flags={row['review_reasons']}; no independent external verification"
        )
        row["required_followup_action"] = part23_followup(row["review_reasons"])
        row["assessment_rule_version"] = PART23_RULE_VERSION
        row["quality_assessment_status"] = "provisional_unreviewed"
        row["accepted_rcid"] = pd.NA
        row["human_review_status"] = "unreviewed"
        rows.append(row)
    result = pd.DataFrame(rows)
    for column in ("gvkey", "rcid", "accepted_rcid"):
        result[column] = result[column].astype("string")
    return result


#### 2.3.4 Firm review and priority rules

Definitions only. The priority table retains one row per flagged pair, plus a row
for each unmatched firm. One row may carry several reasons. No leader is assigned
to a multiple-RCID firm; all alternatives remain in the firm table.


In [4]:
# 2.3.4 — firm review and overlapping priority tables (definitions only)
def part23_assess_firms(firms, candidates):
    groups = {g: f for g, f in candidates.groupby("gvkey", sort=False)}
    rows = []
    for original in firms.to_dict("records"):
        row = dict(original)
        group = groups.get(row["gvkey"], candidates.iloc[:0])
        reasons = tuple(sorted({reason for values in group.review_reasons for reason in values}))
        categories = tuple(group.review_category)
        category = (
            "Unmatched" if group.empty else
            "Conflicting evidence" if "Conflicting evidence" in categories else
            "Ambiguous multiple-RCID candidate" if len(group) > 1 else categories[0]
        )
        if group.empty:
            reasons = ("unmatched",)
        row.update(
            review_category=category, review_reasons=reasons,
            matching_methods=tuple(sorted({m for values in group.evidence_methods for m in values})),
            evidence_origins=tuple(sorted({o for values in group.evidence_origins for o in values})),
            evidence_ids=tuple(i for values in group.evidence_ids for i in values),
            candidate_assessments=tuple(
                dict(rcid=r.rcid, category=r.review_category, methods=r.supporting_identifier_types,
                     reasons=r.review_reasons, evidence_ids=r.evidence_ids)
                for r in group.itertuples(index=False)
            ),
            historical_only_candidate_count=int(group.historical_only.sum()),
            shared_rcid_flag=bool(group.rcid_shared_across_targets.any()),
            identifier_conflict_flag=bool(group.identifier_conflict_flag.any()),
            hierarchy_scope_review=bool(group.hierarchy_scope_review.any()),
            provisional_leading_rcid=group.iloc[0].rcid if category == "Strongly corroborated candidate" else pd.NA,
            provisional_leader_basis="Singleton with corroboration under stated rules; not accepted"
                if category == "Strongly corroborated candidate" else "No leader under conservative rule",
            accepted_rcid=pd.NA, human_review_status="unreviewed",
            historical_validity_status="unverified_snapshot_association",
            independent_external_verification_status="not_performed",
            required_followup_action=part23_followup(reasons),
            assessment_rule_version=PART23_RULE_VERSION,
        )
        rows.append(row)
    result = pd.DataFrame(rows)
    for column in ("gvkey", "accepted_rcid", "provisional_leading_rcid"):
        result[column] = result[column].astype("string")
    return result

def part23_priorities(firms, candidates):
    # Pair-level rows plus the unmatched firm: groups overlap; never add group counts.
    columns = ["gvkey", "rcid", "research_company_names", "revelio_company_names",
               "review_category", "review_reasons", "historical_only",
               "multiple_rcid_flag", "rcid_shared_across_targets", "identifier_conflict_flag",
               "evidence_ids", "required_followup_action"]
    priority = candidates.loc[candidates.review_reasons.map(bool), columns].copy()
    priority["record_type"] = "candidate"
    unmatched = firms.loc[firms.candidate_count.eq(0), [
        "gvkey", "research_company_names", "review_category", "review_reasons",
        "evidence_ids", "required_followup_action"
    ]].copy()
    unmatched["rcid"] = pd.Series(pd.NA, index=unmatched.index, dtype="string")
    unmatched["record_type"] = "unmatched_firm"
    priority = pd.concat([priority, unmatched], ignore_index=True)
    priority["priority_band"] = priority.review_reasons.map(
        lambda reasons: "1 - resolve identity conflict / absence"
        if set(reasons) & {"shared_rcid", "identifier_conflict", "unmatched"}
        else "2 - resolve alternatives / historical-only"
        if set(reasons) & {"multiple_rcids", "historical_only"}
        else "3 - resolve remaining evidence questions"
    )
    return priority.sort_values(["priority_band", "gvkey", "rcid"], na_position="last").reset_index(drop=True)


#### 2.3.5 Synthetic checks

Tiny artificial examples test conflicts, overlapping flags, zero preservation,
CUSIP validity/format, hierarchy scope, missing evidence, and provisional-only
leaders. These tests neither read nor change research source files.


In [5]:
# 2.3.5 — synthetic rule checks only; no research files are read or written
def part23_test_rules():
    assert callable(globals().get("part23_assess_candidates")), "Part 2.3 assessment not implemented"
    # Padding survives; comparison-only CIK normalization and full-CUSIP rules.
    assert part23_compare(("000123",), ("123",), matching_workflow.positive_digits)[0] == "agreement"
    assert part23_compare(("03783310",), ("037833100",), part23_cusip_key)[0] == "disagreement_all_observed"
    assert part23_compare(("000123",), ("123", "456"), matching_workflow.positive_digits)[0] == "mixed_agreement_and_disagreement"
    assert part23_compare((), ("0",), matching_workflow.positive_digits)[0] == "missing_or_invalid"
    assert part23_cusip_key("037833109") == ""
    # Real consolidation summarizer, miniature data, including overlaps and no evidence.
    annual = pd.DataFrame([
        dict(gvkey_key=g, compustat_row=i, conm="Synthetic", cik="000123",
             cusip="037833100", tic="TEST", datadate="2020-12-31")
        for i, g in enumerate(("000001", "000002", "000003", "000004", "000005"), 1)
    ])
    targets = pd.DataFrame({"gvkey": annual.gvkey_key})
    records = []
    specs = [
        ("000001", "001", ("direct_gvkey", "cik"), "full_reference", "037833100", "123"),
        ("000002", "002", ("direct_gvkey", "cik"), "full_reference", "037833100", "123"),
        ("000002", "003", ("ticker",), "earlier_local", "037833100", "123"),
        ("000003", "003", ("cik",), "full_reference", "037833100", "123"),
        ("000004", "004", ("direct_gvkey", "cik"), "full_reference", "594918104", "123"),
    ]
    for row_number, (g, r, methods, origin, cusip, cik) in enumerate(specs, 1):
        for method in methods:
            records.append(dict(
                evidence_id=f"test:{row_number}:{method}", gvkey=g, rcid=r, method=method,
                origin=origin, revelio_source_row=row_number, compustat_rows=(int(g),),
                validated_full_cusip_support=False, comparison_key=g if method == "direct_gvkey" else cik,
                revelio_raw=dict(company="Synthetic", gvkey=g, cik=cik, cusip=cusip,
                                 rcid=r, ultimate_parent_rcid=r, child_rcid=r, ticker="TEST"),
                source_evidence={}, legacy_flags={},
            ))
    ledger = pd.DataFrame(records)
    candidates, firms = matching_workflow.summarize_candidates(targets, annual, ledger)
    before = matching_workflow.state_digest({"c": candidates, "f": firms, "e": ledger})
    assessed = part23_assess_candidates(candidates, ledger, annual)
    reviewed = part23_assess_firms(firms, assessed)
    keyed = assessed.set_index(["gvkey", "rcid"])
    assert keyed.loc[("000001", "001"), "review_category"] == "Strongly corroborated candidate"
    assert keyed.loc[("000002", "002"), "review_category"] == "Ambiguous multiple-RCID candidate"
    assert keyed.loc[("000002", "003"), "review_category"] == "Conflicting evidence"
    assert keyed.loc[("000002", "003"), "historical_only"]
    assert keyed.loc[("000004", "004"), "review_category"] == "Plausible candidate requiring review"
    assert not keyed.loc[("000004", "004"), "identifier_conflict_flag"]  # security discrepancy only
    indexed = reviewed.set_index("gvkey")
    assert indexed.loc["000001", "provisional_leading_rcid"] == "001"
    assert pd.isna(indexed.loc["000002", "provisional_leading_rcid"])
    assert indexed.loc["000002", "candidate_rcids"] == ("002", "003")
    assert indexed.loc["000005", "review_category"] == "Unmatched"
    assert reviewed.accepted_rcid.isna().all() and reviewed.human_review_status.eq("unreviewed").all()
    assert assessed.accepted_rcid.isna().all()
    assert keyed.loc[("000001", "001"), "compustat_cik_raw"] == ("000123",)
    assert before == matching_workflow.state_digest({"c": candidates, "f": firms, "e": ledger})
    priority = part23_priorities(reviewed, assessed)
    assert not priority.duplicated(["gvkey", "rcid"]).any()
    assert set(priority.loc[priority.record_type.eq("unmatched_firm"), "gvkey"]) == {"000005"}
    overlap = priority.loc[priority.gvkey.eq("000002") & priority.rcid.eq("003")].iloc[0]
    assert {"multiple_rcids", "shared_rcid", "historical_only"} <= set(overlap.review_reasons)
    # Missing evidence cannot silently produce a plausible identity.
    no_support = candidates.iloc[[0]].copy()
    no_support["evidence_ids"] = [()]
    no_support["evidence_record_count"] = 0
    weak = part23_assess_candidates(no_support, ledger.iloc[:0], annual)
    assert weak.iloc[0].review_category == "Insufficient evidence"
    # A different parent is a scope question, never an invented corporate history.
    scoped = candidates.iloc[[0]].copy()
    scoped["revelio_ultimate_parent_rcid_raw"] = [("009",)]
    scope_result = part23_assess_candidates(scoped, ledger, annual).iloc[0]
    assert scope_result.hierarchy_scope_review
    assert scope_result.review_category == "Plausible candidate requiring review"
    # A mismatching observed CIK overrides two nominal matching methods.
    conflicted = candidates.iloc[[0]].copy()
    conflicted["revelio_cik_raw"] = [("456",)]
    conflict_result = part23_assess_candidates(conflicted, ledger, annual).iloc[0]
    assert conflict_result.review_category == "Conflicting evidence"
    # Invalid check digit never contributes a validated identifier.
    invalid = candidates.iloc[[0]].copy()
    invalid["revelio_cusip_raw"] = [("037833109",)]
    invalid_result = part23_assess_candidates(invalid, ledger, annual).iloc[0]
    assert invalid_result.identifier_data_quality_issue
    assert invalid_result.review_category == "Plausible candidate requiring review"
    # Ticker alone remains insufficient even with a coincidentally compatible name.
    ticker_ledger = ledger.iloc[[0]].copy()
    ticker_ledger["origin"] = "earlier_local"
    ticker_ledger["method"] = "ticker"
    ticker_candidates, _ = matching_workflow.summarize_candidates(targets.iloc[:1], annual, ticker_ledger)
    ticker_result = part23_assess_candidates(ticker_candidates, ticker_ledger, annual).iloc[0]
    assert ticker_result.review_category == "Insufficient evidence" and ticker_result.historical_only
    # Two methods on different full-reference rows are not co-occurring corroboration.
    split_ledger = ledger.iloc[:2].copy()
    split_ledger["revelio_source_row"] = [1, 99]
    split_candidates, _ = matching_workflow.summarize_candidates(targets.iloc[:1], annual, split_ledger)
    split_result = part23_assess_candidates(split_candidates, split_ledger, annual).iloc[0]
    assert split_result.max_identifiers_on_one_full_reference_row == 1
    assert split_result.review_category == "Plausible candidate requiring review"
    print("PASS: synthetic review rules, overlap handling, identifier strings, and no acceptance.")

part23_test_rules()


PASS: synthetic review rules, overlap handling, identifier strings, and no acceptance.


#### 2.3.6 Build the provisional review tables

Uses only the restored/consolidated evidence already in memory. No files are written.


In [6]:
# 2.3.6 — build provisional tables in memory; no files written
part23_candidate_review = part23_assess_candidates(
    part23_inputs["candidates"], part23_inputs["evidence"], part23_state["compustat_evidence"]
)
part23_firm_review = part23_assess_firms(part23_inputs["firms"], part23_candidate_review)
part23_review_priority = part23_priorities(part23_firm_review, part23_candidate_review)
# Keep the complete ledger for evidence_id -> raw identifiers, source rows and legacy flags.
part23_evidence_review = part23_inputs["evidence"].copy(deep=True)
print("In-memory tables ready for validation:",
      len(part23_firm_review), "firms;", len(part23_candidate_review), "candidate pairs.")


In-memory tables ready for validation: 652 firms; 854 candidate pairs.


#### 2.3.7 First execution checkpoint — STOP after this cell

Expected fixed counts: **652 firms; 651 candidate-covered; 854 pairs; 780 full-reference;
74 historical-only pairs across 44 firms; 151 multiple-RCID firms; one distinct shared
RCID; one unmatched firm (179700); zero accepted; 652 unreviewed**.

Category totals, detected identifier conflicts, priority rows, and provisional leader
counts are computed from the data. Return the validation table, category and priority
summaries, leader count, shared-RCID rows, unmatched row, and final PASS messages.
If an assertion fails, return its traceback and stop; do not relax the checks.


In [7]:
# 2.3.7 — FIRST PART 2.3 EXECUTION CHECKPOINT; stop after this cell
p23c, p23f, p23e = part23_candidate_review, part23_firm_review, part23_evidence_review
p23m = part23_inputs["membership"]
p23pairs = lambda frame: set(frame[["gvkey", "rcid"]].itertuples(index=False, name=None))
assert len(p23f) == p23f.gvkey.nunique() == 652
assert len(p23c) == len(p23pairs(p23c)) == 854
assert list(p23f.gvkey) == list(part23_inputs["firms"].gvkey)
assert p23pairs(p23c) == p23pairs(part23_inputs["candidates"])
assert p23pairs(p23c.loc[~p23c.historical_only]) == p23pairs(p23m.loc[p23m.in_full_reference])
assert p23f.candidate_count.sum() == len(p23c)
assert set(p23f.loc[p23f.candidate_count.eq(0), "gvkey"]) == {"179700"}
assert p23e.evidence_id.is_unique
p23lookup = p23e.set_index("evidence_id")
p23linked = []
for row in p23c.itertuples(index=False):
    assert all(isinstance(x, str) and x for x in (row.gvkey, row.rcid))
    assert len(row.evidence_ids) == len(set(row.evidence_ids)) == row.evidence_record_count
    links = p23lookup.loc[list(row.evidence_ids)]
    assert links.gvkey.eq(row.gvkey).all() and links.rcid.eq(row.rcid).all()
    assert row.supporting_identifier_types == part23_exact_methods(links)
    assert row.historical_only == (not links.origin.eq("full_reference").any())
    p23linked.extend(row.evidence_ids)
assert len(p23linked) == len(p23e) and set(p23linked) == set(p23e.evidence_id)
# Original raw identifiers, names, alternatives, dates and evidence references survive exactly.
p23_original_columns = [c for c in part23_inputs["candidates"] if c != "quality_assessment_status"]
pd.testing.assert_frame_equal(
    p23c[p23_original_columns], part23_inputs["candidates"][p23_original_columns], check_dtype=False
)
pd.testing.assert_series_equal(p23f.candidate_rcids, part23_inputs["firms"].candidate_rcids)
pd.testing.assert_frame_equal(p23e, part23_inputs["evidence"])
assert matching_workflow.state_digest(part23_inputs) == part23_inputs_digest
assert matching_workflow.state_digest(part23_state) == part23_state_digest
for frame in (p23c, p23f):
    assert frame.accepted_rcid.isna().all()
    assert frame.human_review_status.eq("unreviewed").all()
    assert frame.historical_validity_status.eq("unverified_snapshot_association").all()
    assert frame.independent_external_verification_status.eq("not_performed").all()
    assert frame.required_followup_action.str.len().gt(0).all()
leaders = p23f.loc[p23f.provisional_leading_rcid.notna()]
assert leaders.candidate_count.eq(1).all()
assert leaders.review_category.eq("Strongly corroborated candidate").all()
assert all(r.provisional_leading_rcid in r.candidate_rcids for r in leaders.itertuples())
assert not part23_review_priority.duplicated(["gvkey", "rcid"]).any()
priority_pairs = p23pairs(part23_review_priority.loc[part23_review_priority.record_type.eq("candidate")])
required_pairs = p23pairs(p23c.loc[
    p23c.multiple_rcid_flag | p23c.historical_only |
    p23c.rcid_shared_across_targets | p23c.identifier_conflict_flag
])
assert required_pairs <= priority_pairs
assert set(part23_review_priority.loc[
    part23_review_priority.record_type.eq("unmatched_firm"), "gvkey"]) == {"179700"}

part23_validation = pd.DataFrame([
    ("target_firms", len(p23f), 652),
    ("candidate_covered_firms", int(p23f.candidate_count.gt(0).sum()), 651),
    ("candidate_pairs", len(p23c), 854),
    ("full_reference_pairs", int((~p23c.historical_only).sum()), 780),
    ("historical_only_pairs", int(p23c.historical_only.sum()), 74),
    ("historical_only_firms", p23c.loc[p23c.historical_only, "gvkey"].nunique(), 44),
    ("multiple_rcid_firms", int(p23f.multiple_rcid_flag.sum()), 151),
    ("shared_distinct_rcids", p23c.loc[p23c.rcid_shared_across_targets, "rcid"].nunique(), 1),
    ("unmatched_firms", int(p23f.candidate_count.eq(0).sum()), 1),
    ("accepted_firm_identities", int(p23f.accepted_rcid.notna().sum()), 0),
    ("unreviewed_firms", int(p23f.human_review_status.eq("unreviewed").sum()), 652),
], columns=["check", "observed", "expected"])
assert part23_validation.observed.eq(part23_validation.expected).all()
part23_validation["status"] = "PASS"
part23_category_summary = pd.concat([
    p23f.review_category.value_counts().rename("firms"),
    p23c.review_category.value_counts().rename("candidate_pairs"),
], axis=1).fillna(0).astype(int)
p23exploded = part23_review_priority.explode("review_reasons")
part23_priority_summary = p23exploded.groupby("review_reasons").agg(
    firms=("gvkey", "nunique"), candidate_pairs=("rcid", "count")
).sort_index()
part23_meeting_summary = f"""# Provisional Revelio firm-matching review
- Validated universe: 652 target GVKEY firms; 651 have candidates (99.85%).
  All 854 candidate pairs remain: 780 full-reference and 74 historical-only pairs
  across 44 firms. 'Historical-only' means absent from full-reference matching;
  it does not establish historical correctness.
- Review priorities overlap: 151 firms have multiple RCIDs, one distinct RCID is
  shared across targets, and GVKEY 179700 is unmatched. Additional identifier,
  security-format, scope and evidence gaps are in the priority table.
- Methods: exact trimmed GVKEY, normalized CIK and validated full-CUSIP support;
  original ticker/shape-only evidence remains visible. Method counts and repeated
  Revelio records are not independent verification.
- Categories describe evidence strength. {len(leaders)} singleton provisional
  leaders meet the documented conservative rule; accepted identities remain zero.
  All 652 firms are unreviewed; alternatives are retained.
- The 2018–2025 relationship remains unverified for every candidate. Reference
  snapshot associations and dated source observations do not establish effective
  entity/parent relationships for each study year. No corporate events are inferred.
- Next: inspect conflicts, alternatives and historical-only cases; perform external
  dated entity research; document human decisions separately. After this checkpoint
  is reviewed, prepare a new versioned CSV/Parquet and Markdown package with the
  evidence ledger, rule definitions and checkpoint provenance.
"""
display(part23_validation)
display(part23_category_summary)
display(part23_priority_summary)
print("Provisional leaders (not accepted):", len(leaders))
print("Priority table:", len(part23_review_priority), "rows;",
      part23_review_priority.gvkey.nunique(), "distinct firms. Groups overlap.")
with pd.option_context("display.max_columns", None, "display.max_colwidth", 90):
    display(p23c.loc[p23c.rcid_shared_across_targets, [
        "gvkey", "rcid", "research_company_names", "revelio_company_names",
        "review_category", "review_reasons", "evidence_ids"]])
    display(p23f.loc[p23f.gvkey.eq("179700"), [
        "gvkey", "research_company_names", "candidate_count", "review_category",
        "accepted_rcid", "human_review_status", "required_followup_action"]])
    display(part23_review_priority.head(12))
print(part23_meeting_summary)
print("PASS: Part 2.3 baseline, uniqueness, evidence traceability and preservation checks.")
print("Accepted identities: 0. Files exported: 0. Historical identities remain unverified.")
print("STOP: return validation, category/priority summaries, shared-RCID rows and unmatched row.")


,check,observed,expected,status
0,target_firms,652,652,PASS
1,candidate_covered_firms,651,651,PASS
2,candidate_pairs,854,854,PASS
3,full_reference_pairs,780,780,PASS
4,historical_only_pairs,74,74,PASS
5,historical_only_firms,44,44,PASS
6,multiple_rcid_firms,151,151,PASS
7,shared_distinct_rcids,1,1,PASS
8,unmatched_firms,1,1,PASS
9,accepted_firm_identities,0,0,PASS


,firms,candidate_pairs
review_category,,
Plausible candidate requiring review,473,473
Ambiguous multiple-RCID candidate,88,265
Conflicting evidence,88,114
Strongly corroborated candidate,2,2
Unmatched,1,0


,firms,candidate_pairs
review_reasons,,
cusip_discrepancy,100,135
hierarchy_scope,649,816
historical_only,44,74
identifier_conflict,87,113
insufficient_identifiers,43,73
legacy_name_ticker_link_concern,37,63
missing_identifier_evidence,156,210
multiple_rcids,151,354
no_dated_window_evidence,11,11


Provisional leaders (not accepted): 2
Priority table: 853 rows; 650 distinct firms. Groups overlap.


,gvkey,rcid,research_company_names,revelio_company_names,review_category,review_reasons,evidence_ids
385,012886,430242,"(TWENTY-FIRST CENTURY FOX INC,)","(Fox Corp.,)",Conflicting evidence,"(identifier_conflict, shared_rcid, multiple_rcids, historical_only, hierarchy_scope, c...","(legacy:243,)"
586,034636,430242,"(FOX CORP,)","(Fox Corp.,)",Conflicting evidence,"(shared_rcid, hierarchy_scope)","(full:449, full:450, full:451, v1:034636:430242:cik:456, v1:034636:430242:direct_gvkey..."


,gvkey,research_company_names,candidate_count,review_category,accepted_rcid,human_review_status,required_followup_action
625,179700,(),0,Unmatched,<NA>,unreviewed,Verify entity scope and dated RCID relationship for 2018–2025 using external sources; ...


,gvkey,rcid,research_company_names,revelio_company_names,review_category,review_reasons,historical_only,multiple_rcid_flag,rcid_shared_across_targets,identifier_conflict_flag,evidence_ids,required_followup_action,record_type,priority_band
0,001380,22593742,"(HESS CORP,)","(Hydraulic Elements & Systems AD,)",Conflicting evidence,"(identifier_conflict, multiple_rcids, historical_only, cusip_discrepancy, missing_iden...",True,True,False,True,"(legacy:1, legacy:2, legacy:3, legacy:4, legacy:5, legacy:6, legacy:7, legacy:205)",Verify entity scope and dated RCID relationship for 2018–2025 using external sources; ...,candidate,1 - resolve identity conflict / absence
1,001598,73989,"(AMETEK INC,)","(AMETEK, Inc.,)",Conflicting evidence,"(identifier_conflict, hierarchy_scope)",False,False,False,True,"(full:62, full:63, v1:001598:73989:direct_gvkey:79, v1:001598:73989:full_cusip:79)",Verify entity scope and dated RCID relationship for 2018–2025 using external sources; ...,candidate,1 - resolve identity conflict / absence
2,002019,1504569,"(BANK OF NEW YORK MELLON CORP,)","(The Bank of New York Mellon Corp.,)",Conflicting evidence,"(identifier_conflict, hierarchy_scope)",False,False,False,True,"(full:1575, full:1576)",Verify entity scope and dated RCID relationship for 2018–2025 using external sources; ...,candidate,1 - resolve identity conflict / absence
3,002154,217756,"(BEMIS CO INC,)","(Braemar Plc,)",Conflicting evidence,"(identifier_conflict, multiple_rcids, historical_only, hierarchy_scope, cusip_discrepa...",True,True,False,True,"(legacy:8, legacy:206)",Verify entity scope and dated RCID relationship for 2018–2025 using external sources; ...,candidate,1 - resolve identity conflict / absence
4,002176,20948983,"(BERKSHIRE HATHAWAY,)","(Brookside Energy Ltd.,)",Conflicting evidence,"(identifier_conflict, multiple_rcids, historical_only, hierarchy_scope, cusip_discrepa...",True,True,False,True,"(legacy:210,)",Verify entity scope and dated RCID relationship for 2018–2025 using external sources; ...,candidate,1 - resolve identity conflict / absence
5,002176,292851,"(BERKSHIRE HATHAWAY,)","(Brooks MacDonald Group Plc,)",Conflicting evidence,"(identifier_conflict, multiple_rcids, historical_only, hierarchy_scope, cusip_discrepa...",True,True,False,True,"(legacy:208,)",Verify entity scope and dated RCID relationship for 2018–2025 using external sources; ...,candidate,1 - resolve identity conflict / absence
6,002435,1098506,"(BROWN FORMAN CORP,)","(Bolognafiere SPA,)",Conflicting evidence,"(identifier_conflict, multiple_rcids, historical_only, hierarchy_scope, cusip_discrepa...",True,True,False,True,"(legacy:211,)",Verify entity scope and dated RCID relationship for 2018–2025 using external sources; ...,candidate,1 - resolve identity conflict / absence
7,003231,802826,"(COMERICA INC,)","(Ciments du Maroc SA,)",Conflicting evidence,"(identifier_conflict, multiple_rcids, historical_only, hierarchy_scope, cusip_discrepa...",True,True,False,True,"(legacy:10, legacy:12, legacy:14, legacy:16, legacy:18, legacy:20, legacy:22, legacy:2...",Verify entity scope and dated RCID relationship for 2018–2025 using external sources; ...,candidate,1 - resolve identity conflict / absence
8,003310,639602,"(CA INC,)","(Carrefour SA,)",Conflicting evidence,"(identifier_conflict, multiple_rcids, historical_only, hierarchy_scope, cusip_discrepa...",True,True,False,True,"(legacy:214,)",Verify entity scope and dated RCID relationship for 2018–2025 using external sources; ...,candidate,1 - resolve identity conflict / absence
9,003980,745033,"(DISNEY (WALT) CO,)","(The Walt Disney Co.,)",Conflicting evidence,"(identifier_conflict, hierarchy_scope)",False,False,False,True,"(full:793, full:794, v1:003980:745033:direct_gvkey:343, v1:003980:745033:full_cusip:343)",Verify entity scope and dated RCID relationship for 2018–2025 using external sources; ...,candidate,1 - resolve identity conflict / absence


# Provisional Revelio firm-matching review
- Validated universe: 652 target GVKEY firms; 651 have candidates (99.85%).
  All 854 candidate pairs remain: 780 full-reference and 74 historical-only pairs
  across 44 firms. 'Historical-only' means absent from full-reference matching;
  it does not establish historical correctness.
- Review priorities overlap: 151 firms have multiple RCIDs, one distinct RCID is
  shared across targets, and GVKEY 179700 is unmatched. Additional identifier,
  security-format, scope and evidence gaps are in the priority table.
- Methods: exact trimmed GVKEY, normalized CIK and validated full-CUSIP support;
  original ticker/shape-only evidence remains visible. Method counts and repeated
  Revelio records are not independent verification.
- Categories describe evidence strength. 2 singleton provisional
  leaders meet the documented conservative rule; accepted identities remain zero.
  All 652 firms are unreviewed; alternatives are retained.
- The 2018–2025 rela

### 2.3 Interpretability audit (v2) — run 2.3.8–2.3.12, then STOP

**The executed 2.3.1–2.3.7 cells and outputs above are retained as the v1 audit
baseline, not the final review rules.** The new cells keep separate v1/v2 tables.
On the current kernel, run only the five new code cells below. On a fresh kernel,
first run 2.3.1–2.3.7 to restore the selected checkpoint and reproduce v1.
No reference scan or export is needed.

**Observed defect and minimal correction.** The v1 hierarchy flag covered 816
pairs / 649 firms: different children occurred for 785 pairs / 648 firms, and
different parents for 195 pairs / 166 firms. These are overlapping metadata
observations, not demonstrated identity conflicts. Ordinary parent/child fields
must not veto identifier corroboration. v2 separates metadata presence, a different
parent requiring historical scope research, and multiple recorded parent values
requiring reconciliation. The last is a metadata inconsistency, not proof of an
incorrect identity or ownership history. No candidate pair has multiple parent
values in this checkpoint. These checks concern only the retained records, not
the completeness or correctness of the provider's hierarchy.

**Why only two leaders?** Of 500 singleton candidates, 471 satisfy all non-hierarchy
corroboration conditions. v1 excluded 469 solely for parent/child metadata, leaving
GVKEY 024617 / RCID 1101337 (Old Dominion Freight) and GVKEY 061380 / RCID 22142228
(Erie Indemnity). Removing that mistaken veto produces 471 provisional leaders,
including 25 with a different recorded parent. Those 25 explicitly retain a
parent-scope research task. The two-identifier, same-full-reference-row threshold,
singleton requirement, identifier conflicts, shared-RCID exclusion, and other
evidence requirements remain unchanged. Leaders are priorities for review, not
accepted identities or validated historical entity relationships.

**CUSIP audit.** All nonblank observed CUSIPs are valid nine-character strings.
There are 639 exact-agreement pairs, 135 disjoint-security pairs (100 firms), and
80 non-comparable pairs with missing evidence. The 135 comprise 72 full-reference
and 63 historical-only pairs. There are zero observed 8/9 format cases, invalid
CUSIPs, or within-target CUSIP variation in this retained annual evidence.
Consequently no historical security change is demonstrated here. v2 separates
availability/validity, exact agreement, valid 8/9 format equivalence, and unresolved
security differences. Format equivalence does not create a matching method.
Disjoint CUSIPs remain review issues, never sufficient on their own to declare an
entity contradiction. This refinement changes no categories on this checkpoint.

**Expected before/after firm counts (audit results, not classification targets):**

| Category | v1 | v2 |
|---|---:|---:|
| Strongly corroborated candidate | 2 | 471 |
| Plausible candidate requiring review | 473 | 4 |
| Ambiguous multiple-RCID candidate | 88 | 88 |
| Conflicting evidence | 88 | 88 |
| Unmatched | 1 | 1 |

The four remaining plausible singleton candidates (010466, 025056, 025356, 065967)
have direct-GVKEY support only and no dated study-window source evidence.
Conflict categories still take precedence: 63 of the 151 multiple-RCID firms fall
under Conflicting evidence, leaving 88 in the multiple-RCID category.

**Examples and limits.** The next displays include Pinnacle West (ordinary child
metadata); Hess → Hydraulic Elements & Systems AD (ticker-only HES association
with a different GVKEY); Franklin Templeton → Digidesign / Divi Hotels Marketing
(CIK reuse/inconsistency in the reference, not independently verified associations);
both GVKEYs sharing Fox RCID 430242; AMETEK (matching name/GVKEY/CUSIP but discrepant
CIK); and American Airlines → AMR (different CUSIP and multiple alternatives).
Names raise review questions; no corporate event or identity correctness is inferred.
All examples retain evidence IDs and original source-row links.


In [8]:
# 2.3.8 — preserve v1 results and verify the audit inputs; no writes
part23_audit_required = (
    "part23_candidate_review", "part23_firm_review", "part23_evidence_review",
    "part23_inputs", "part23_state", "part23_inputs_digest", "part23_state_digest",
)
assert all(name in globals() for name in part23_audit_required), (
    "Run 2.3.1–2.3.7 first on a fresh kernel; do not rerun the reference scan."
)
assert part23_candidate_review.assessment_rule_version.eq("revelio-review-v1-2026-11-08").all()
part23_v1_candidates = part23_candidate_review.copy(deep=True)
part23_v1_firms = part23_firm_review.copy(deep=True)
part23_v1_digest = matching_workflow.state_digest({
    "c": part23_v1_candidates, "f": part23_v1_firms,
})
assert matching_workflow.state_digest(part23_inputs) == part23_inputs_digest
assert matching_workflow.state_digest(part23_state) == part23_state_digest
assert part23_v1_firms.review_category.value_counts().to_dict() == {
    "Plausible candidate requiring review": 473,
    "Ambiguous multiple-RCID candidate": 88, "Conflicting evidence": 88,
    "Strongly corroborated candidate": 2, "Unmatched": 1,
}, "The input is not the validated v1 run; inspect before continuing."
print("PASS: v1 audit baseline preserved; no files written.")


PASS: v1 audit baseline preserved; no files written.


#### 2.3.9 Corrected diagnostics and review rules

Definitions only. v1 fields remain available beside v2 interpretations. Mere parent/child metadata is informational; a different parent retains an explicit research task.


In [9]:
# 2.3.9 — minimal v2 corrections and diagnostic definitions
PART23_AUDIT_RULE_VERSION = "revelio-review-v2-hierarchy-cusip-audit"

def part23_hierarchy_audit(parents, children, rcid):
    clean = lambda values: tuple(sorted({matching_workflow.text_key(v) for v in values} - {""}))
    p, ch = clean(parents), clean(children)
    return {
        "hierarchy_metadata_observed": bool(p or ch),
        "different_parent_metadata_observed": bool(set(p) - {rcid}),
        "different_child_metadata_observed": bool(set(ch) - {rcid}),
        "parent_scope_research_required": bool(set(p) - {rcid}),
        "hierarchy_metadata_inconsistency": len(p) > 1,
        "hierarchy_scope_review": len(p) > 1,
        "hierarchy_evidence_status": (
            "multiple_parent_values_unresolved" if len(p) > 1 else
            "metadata_only_no_identity_conflict_demonstrated" if p or ch else
            "no_hierarchy_metadata"
        ),
        "hierarchy_identity_conflict_status": "not_established_by_metadata",
    }

def part23_cusip_audit(target_values, reference_values):
    def side(values):
        raw = {matching_workflow.text_key(v) for v in values} - {""}
        valid = {part23_cusip_key(v) for v in raw} - {""}
        invalid = tuple(sorted(v for v in raw if not part23_cusip_key(v)))
        availability = ("missing" if not raw else "invalid_only" if not valid else
                        "valid_and_invalid" if invalid else "valid")
        return valid, invalid, availability
    a, bad_a, avail_a = side(target_values)
    b, bad_b, avail_b = side(reference_values)
    a8, b8 = {v[:8] for v in a}, {v[:8] for v in b}
    relation = (
        "not_comparable" if not a or not b else
        "full_string_agreement" if b <= a else
        "format_equivalence_only" if b8 <= a8 else
        "mixed_security_values_unresolved" if a8 & b8 else
        "different_security_values_unresolved"
    )
    return {
        "target_cusip_availability": avail_a, "reference_cusip_availability": avail_b,
        "target_invalid_cusips": bad_a, "reference_invalid_cusips": bad_b,
        "cusip_relation": relation,
        "cusip_security_difference": relation in {
            "mixed_security_values_unresolved", "different_security_values_unresolved"
        },
        "cusip_format_equivalence": relation == "format_equivalence_only",
        "target_security_variation_observed": len(a8) > 1,
        "cusip_entity_contradiction_status": "not_established_by_cusip_comparison",
        "historical_security_change_status": "requires_dated_external_research",
    }

def part23_apply_audited_rules(v1_candidates):
    rows = []
    for original in v1_candidates.to_dict("records"):
        row = dict(original)
        row["v1_review_category"] = row["review_category"]
        row["v1_review_reasons"] = row["review_reasons"]
        row["v1_hierarchy_scope_review"] = row["hierarchy_scope_review"]
        row["v1_cusip_discrepancy_flag"] = row["cusip_discrepancy_flag"]
        row.update(part23_hierarchy_audit(
            row["revelio_ultimate_parent_rcid_raw"], row["revelio_child_rcid_raw"], row["rcid"]))
        row.update(part23_cusip_audit(row["compustat_cusip_raw"], row["revelio_cusip_raw"]))
        row["cusip_discrepancy_flag"] = row["cusip_security_difference"]
        row["v1_legacy_name_ticker_link_concern"] = row["legacy_name_ticker_link_concern"]
        row["legacy_name_ticker_link_concern"] = bool(set(row["legacy_true_flags"]) & {
            "name_disagreement", "ticker_reuse_or_conflict", "ls_link_review"
        })
        # Keep exactly the old identifier requirement and all non-format/non-metadata exclusions.
        row["strong_snapshot_corroboration"] = bool(
            row["max_identifiers_on_one_full_reference_row"] >= 2
            and not any((
                row["identifier_conflict_flag"], row["rcid_shared_across_targets"],
                row["cusip_security_difference"], row["identifier_data_quality_issue"],
                row["hierarchy_metadata_inconsistency"], row["legacy_name_ticker_link_concern"],
                row["historical_identifier_variation_flag"], row["no_dated_window_evidence"],
            ))
        )
        reasons = [r for r in row["review_reasons"] if r not in {
            "hierarchy_scope", "cusip_discrepancy", "cusip_format_overlap", "legacy_name_ticker_link_concern"
        }]
        if row["legacy_name_ticker_link_concern"]:
            reasons.append("legacy_name_ticker_link_concern")
        if row["parent_scope_research_required"]:
            reasons.append("parent_scope_research")  # research task, not a corroboration veto
        if row["hierarchy_metadata_inconsistency"]:
            reasons.append("hierarchy_metadata_inconsistency")
        if row["cusip_security_difference"]:
            reasons.append("cusip_security_difference")
        if row["cusip_format_equivalence"]:
            reasons.append("cusip_format_equivalence")  # no new match or identity conflict
        row["review_reasons"] = tuple(reasons)
        row["review_category"] = part23_category(row)
        row["rule_explanation"] = (
            f"{row['review_category']}; exact types={row['supporting_identifier_types']}; "
            f"co-occurring full-reference types={row['max_identifiers_on_one_full_reference_row']}; "
            f"hierarchy={row['hierarchy_evidence_status']}; CUSIP={row['cusip_relation']}; "
            f"review reasons={row['review_reasons']}; automated snapshot corroboration only"
        )
        row["required_followup_action"] = part23_audit_followup(row["review_reasons"])
        row["assessment_rule_version"] = PART23_AUDIT_RULE_VERSION
        rows.append(row)
    result = pd.DataFrame(rows)
    for col in ("gvkey", "rcid", "accepted_rcid"):
        result[col] = result[col].astype("string")
    assert result.accepted_rcid.isna().all()
    assert result.human_review_status.eq("unreviewed").all()
    return result

def part23_audit_followup(reasons):
    action = part23_followup(reasons)
    if "parent_scope_research" in reasons:
        action += "; Verify the recorded parent and historical entity scope; metadata alone is not a conflict"
    if "hierarchy_metadata_inconsistency" in reasons:
        action += "; Reconcile multiple recorded parent values and source dates; do not infer a transition"
    if "cusip_security_difference" in reasons:
        action += "; Research unresolved CUSIP/security differences and dates; no entity contradiction inferred"
    if "cusip_format_equivalence" in reasons:
        action += "; Record valid 8/9-character format equivalence separately from exact matching"
    return action


#### 2.3.10 Regression checks and v2 tables

Tests cover the real hierarchy regression plus synthetic format, missingness, invalidity, metadata inconsistency, shared-RCID, conflict and multiple-candidate cases. No accepted identities are created.


In [10]:
# 2.3.10 — regression tests, then v2 in-memory tables; no export
def part23_test_audited_rules():
    # A real regression fixture, copied before synthetic alterations.
    template = part23_v1_candidates.set_index(["gvkey", "rcid"], drop=False).loc[
        ("001075", "368391")].to_dict()
    assert template["review_category"] == "Plausible candidate requiring review"
    def assessed(**changes):
        sample = dict(template)
        sample.update(changes)
        return part23_apply_audited_rules(pd.DataFrame([sample])).iloc[0]
    assert assessed().review_category == "Strongly corroborated candidate"
    assert assessed(revelio_ultimate_parent_rcid_raw=("999",)).review_category == "Strongly corroborated candidate"
    assert assessed(revelio_ultimate_parent_rcid_raw=("999", "888")).review_category == "Plausible candidate requiring review"
    assert assessed(identifier_conflict_flag=True).review_category == "Conflicting evidence"
    assert assessed(rcid_shared_across_targets=True).review_category == "Conflicting evidence"
    assert assessed(multiple_rcid_flag=True, candidate_count=2).review_category == "Ambiguous multiple-RCID candidate"
    assert assessed(max_identifiers_on_one_full_reference_row=1).review_category == "Plausible candidate requiring review"
    # Format equivalence is not a validated matching method.
    fmt = assessed(compustat_cusip_raw=("03783310",), revelio_cusip_raw=("037833100",),
                   cusip_discrepancy_flag=True, cusip_8_9_format_overlap=True,
                   supporting_identifier_types=("cik", "direct_gvkey"), supporting_identifier_count=2,
                   max_identifiers_on_one_full_reference_row=2)
    assert fmt.review_category == "Strongly corroborated candidate"
    assert fmt.supporting_identifier_types == ("cik", "direct_gvkey") and not fmt.cusip_security_difference
    assert part23_cusip_audit((), ("037833100",))["target_cusip_availability"] == "missing"
    assert part23_cusip_audit(("037833109",), ("037833100",))["target_cusip_availability"] == "invalid_only"
    assert part23_cusip_audit(("037833100",), ("594918104",))["cusip_security_difference"]
    varied = part23_cusip_audit(("037833100", "594918104"), ("037833100",))
    assert varied["target_security_variation_observed"] and not varied["cusip_security_difference"]
    assert assessed().compustat_cik_raw == template["compustat_cik_raw"]
    assert pd.isna(assessed().accepted_rcid)
    print("PASS: hierarchy/CUSIP regression tests; conflicts and no-acceptance safeguards retained.")

part23_test_audited_rules()
part23_v2_candidates = part23_apply_audited_rules(part23_v1_candidates)
part23_v2_firms = part23_assess_firms(part23_inputs["firms"], part23_v2_candidates)
part23_v2_firms["assessment_rule_version"] = PART23_AUDIT_RULE_VERSION
part23_v2_firms["required_followup_action"] = part23_v2_firms.review_reasons.map(part23_audit_followup)
for part23_flag in ("hierarchy_metadata_observed", "different_parent_metadata_observed",
                   "different_child_metadata_observed", "parent_scope_research_required",
                   "hierarchy_metadata_inconsistency", "cusip_security_difference"):
    part23_flag_by_firm = part23_v2_candidates.groupby("gvkey")[part23_flag].any()
    part23_v2_firms[part23_flag] = part23_v2_firms.gvkey.map(part23_flag_by_firm).fillna(False).astype(bool)
part23_v2_priority = part23_priorities(part23_v2_firms, part23_v2_candidates)
print("v2 tables built in memory; v1 tables and all evidence retained.")


PASS: hierarchy/CUSIP regression tests; conflicts and no-acceptance safeguards retained.
v2 tables built in memory; v1 tables and all evidence retained.


#### 2.3.11 Audit reports and illustrative cases

The example selection is intentional, not representative. All records and alternatives remain available. Full raw evidence is linked through evidence IDs; no external corporate history has been asserted.


In [11]:
# 2.3.11 — before/after distributions, gate audit, and traceable real examples
part23_before_after = pd.concat([
    part23_v1_firms.review_category.value_counts().rename("v1_firms"),
    part23_v2_firms.review_category.value_counts().rename("v2_firms"),
    part23_v1_candidates.review_category.value_counts().rename("v1_pairs"),
    part23_v2_candidates.review_category.value_counts().rename("v2_pairs"),
], axis=1).fillna(0).astype(int)
part23_transitions = pd.crosstab(part23_v1_firms.review_category, part23_v2_firms.review_category)
part23_nonhierarchy_exclusions = [
    "identifier_conflict_flag", "rcid_shared_across_targets", "cusip_discrepancy_flag",
    "cusip_8_9_format_overlap", "identifier_data_quality_issue",
    "legacy_name_ticker_link_concern", "historical_identifier_variation_flag", "no_dated_window_evidence",
]
part23_singleton = part23_v1_candidates.candidate_count.eq(1)
part23_other_gates_pass = (
    part23_singleton & part23_v1_candidates.max_identifiers_on_one_full_reference_row.ge(2)
    & ~part23_v1_candidates[part23_nonhierarchy_exclusions].any(axis=1)
)
part23_gate_audit = pd.DataFrame([
    ("singleton candidates", int(part23_singleton.sum())),
    ("pass all non-hierarchy gates", int(part23_other_gates_pass.sum())),
    ("excluded solely by old hierarchy gate", int((part23_other_gates_pass & part23_v1_candidates.hierarchy_scope_review).sum())),
    ("old provisional leaders", int(part23_v1_firms.provisional_leading_rcid.notna().sum())),
    ("v2 provisional leaders", int(part23_v2_firms.provisional_leading_rcid.notna().sum())),
    ("v2 leaders with different recorded parent", int((part23_v2_firms.provisional_leading_rcid.notna() & part23_v2_firms.parent_scope_research_required).sum())),
], columns=["gate", "firms"])
part23_hierarchy_audit_summary = pd.DataFrame([
    (flag, int(part23_v2_candidates[flag].sum()),
     part23_v2_candidates.loc[part23_v2_candidates[flag], "gvkey"].nunique())
    for flag in ("v1_hierarchy_scope_review", "different_parent_metadata_observed",
                 "different_child_metadata_observed", "hierarchy_metadata_inconsistency",
                 "parent_scope_research_required")
], columns=["diagnostic", "pairs", "firms"])
part23_cusip_audit_summary = part23_v2_candidates.groupby([
    "cusip_relation", "target_cusip_availability", "reference_cusip_availability"
]).agg(pairs=("rcid", "size"), firms=("gvkey", "nunique"))
part23_v2_priority_summary = part23_v2_priority.explode("review_reasons").groupby(
    "review_reasons"
).agg(firms=("gvkey", "nunique"), candidate_pairs=("rcid", "count"))

# Deliberately selected audit examples, not a representative sample or human decisions.
part23_example_keys = {
    ("001075", "368391"): "Ordinary child metadata; exact identifiers corroborate the candidate",
    ("001380", "22593742"): "HES ticker-only historical association; names/GVKEY differ",
    ("004885", "19579259"): "Digidesign carries the target CIK; name/CUSIP association needs research",
    ("004885", "480912"): "Divi Hotels carries target CIK but a different GVKEY",
    ("004885", "911388"): "Franklin Templeton alternative retained alongside questionable associations",
    ("012886", "430242"): "Historical FOXA association with shared RCID and conflicting identifiers",
    ("034636", "430242"): "Three identifier methods, but shared RCID still blocks a leader",
    ("001598", "73989"): "AMETEK name/GVKEY/CUSIP agree while normalized CIK disagrees",
    ("001045", "589078"): "AMR CIK association with a different security; historical relationship unverified",
}
part23_example_columns = [
    "gvkey", "rcid", "research_company_names", "revelio_company_names",
    "evidence_methods", "evidence_origins", "review_gvkey_comparison", "review_cik_comparison",
    "compustat_cik_raw", "revelio_cik_raw", "compustat_cusip_raw", "revelio_cusip_raw",
    "cusip_relation", "revelio_ultimate_parent_rcid_raw", "revelio_child_rcid_raw",
    "v1_review_category", "review_category", "historical_only", "evidence_ids",
    "revelio_source_references", "required_followup_action",
]
part23_audit_examples = part23_v2_candidates.set_index(["gvkey", "rcid"], drop=False).loc[
    list(part23_example_keys), part23_example_columns
].reset_index(drop=True)
part23_audit_examples["audit_question"] = [
    part23_example_keys[(r.gvkey, r.rcid)] for r in part23_audit_examples.itertuples()
]
part23_example_ids = {i for ids in part23_audit_examples.evidence_ids for i in ids}
part23_audit_example_evidence = part23_evidence_review.loc[
    part23_evidence_review.evidence_id.isin(part23_example_ids)
].copy()
display(part23_before_after)
display(part23_gate_audit)
display(part23_hierarchy_audit_summary)
display(part23_cusip_audit_summary)
display(part23_v2_priority_summary)
with pd.option_context("display.max_columns", None, "display.max_colwidth", 120):
    display(part23_audit_examples[[
        "gvkey", "rcid", "research_company_names", "revelio_company_names",
        "v1_review_category", "review_category", "audit_question", "evidence_ids"
    ]])
print("Full example details: part23_audit_examples; raw ledger: part23_audit_example_evidence.")


,v1_firms,v2_firms,v1_pairs,v2_pairs
review_category,,,,
Plausible candidate requiring review,473,4,473,4
Ambiguous multiple-RCID candidate,88,88,265,265
Conflicting evidence,88,88,114,114
Strongly corroborated candidate,2,471,2,471
Unmatched,1,1,0,0


,gate,firms
0,singleton candidates,500
1,pass all non-hierarchy gates,471
2,excluded solely by old hierarchy gate,469
3,old provisional leaders,2
4,v2 provisional leaders,471
5,v2 leaders with different recorded parent,25


,diagnostic,pairs,firms
0,v1_hierarchy_scope_review,816,649
1,different_parent_metadata_observed,195,166
2,different_child_metadata_observed,785,648
3,hierarchy_metadata_inconsistency,0,0
4,parent_scope_research_required,195,166


pairs  \
cusip_relation                       target_cusip_availability reference_cusip_availability          
different_security_values_unresolved valid                     valid                           135   
full_string_agreement                valid                     valid                           639   
not_comparable                       missing                   valid                            22   
                                     valid                     missing                          58   

                                                                                             firms  
cusip_relation                       target_cusip_availability reference_cusip_availability         
different_security_values_unresolved valid                     valid                           100  
full_string_agreement                valid                     valid                           639  
not_comparable                       missing                   valid                            11  
                                     valid                     missing                          54

,firms,candidate_pairs
review_reasons,,
cusip_security_difference,100,135
historical_only,44,74
identifier_conflict,87,113
insufficient_identifiers,43,73
legacy_name_ticker_link_concern,36,62
missing_identifier_evidence,156,210
multiple_rcids,151,354
no_dated_window_evidence,11,11
parent_scope_research,166,195


,gvkey,rcid,research_company_names,revelio_company_names,v1_review_category,review_category,audit_question,evidence_ids
0,001075,368391,"(PINNACLE WEST CAPITAL CORP,)","(Pinnacle West Capital Corp.,)",Plausible candidate requiring review,Strongly corroborated candidate,Ordinary child metadata; exact identifiers corroborate the candidate,"(full:372, full:373, full:374, v1:001075:368391:cik:830, v1:001075:368391:direct_gvkey:830, v1:001075:368391:full_cu..."
1,001380,22593742,"(HESS CORP,)","(Hydraulic Elements & Systems AD,)",Conflicting evidence,Conflicting evidence,HES ticker-only historical association; names/GVKEY differ,"(legacy:1, legacy:2, legacy:3, legacy:4, legacy:5, legacy:6, legacy:7, legacy:205)"
2,004885,19579259,"(FRANKLIN TEMPLETON INC,)","(Digidesign, Inc.,)",Ambiguous multiple-RCID candidate,Ambiguous multiple-RCID candidate,Digidesign carries the target CIK; name/CUSIP association needs research,"(full:1623,)"
3,004885,480912,"(FRANKLIN TEMPLETON INC,)","(Divi Hotels Marketing, Inc.,)",Conflicting evidence,Conflicting evidence,Divi Hotels carries target CIK but a different GVKEY,"(full:510,)"
4,004885,911388,"(FRANKLIN TEMPLETON INC,)","(Franklin Templeton, Inc.,)",Ambiguous multiple-RCID candidate,Ambiguous multiple-RCID candidate,Franklin Templeton alternative retained alongside questionable associations,"(full:980, full:981, full:982, v1:004885:911388:cik:158, v1:004885:911388:direct_gvkey:158, v1:004885:911388:full_cu..."
5,012886,430242,"(TWENTY-FIRST CENTURY FOX INC,)","(Fox Corp.,)",Conflicting evidence,Conflicting evidence,Historical FOXA association with shared RCID and conflicting identifiers,"(legacy:243,)"
6,034636,430242,"(FOX CORP,)","(Fox Corp.,)",Conflicting evidence,Conflicting evidence,"Three identifier methods, but shared RCID still blocks a leader","(full:449, full:450, full:451, v1:034636:430242:cik:456, v1:034636:430242:direct_gvkey:456, v1:034636:430242:full_cu..."
7,001598,73989,"(AMETEK INC,)","(AMETEK, Inc.,)",Conflicting evidence,Conflicting evidence,AMETEK name/GVKEY/CUSIP agree while normalized CIK disagrees,"(full:62, full:63, v1:001598:73989:direct_gvkey:79, v1:001598:73989:full_cusip:79)"
8,001045,589078,"(AMERICAN AIRLINES GROUP INC,)","(AMR Corp.,)",Ambiguous multiple-RCID candidate,Ambiguous multiple-RCID candidate,AMR CIK association with a different security; historical relationship unverified,"(full:613,)"


Full example details: part23_audit_examples; raw ledger: part23_audit_example_evidence.


#### 2.3.12 Audit validation checkpoint — STOP before exporting

Return the before/after table, gate audit, hierarchy/CUSIP summaries, example table and final validation. Expected change: 469 plausible singleton candidates become strongly corroborated solely because hierarchy metadata is no longer an automatic veto. Zero identities are accepted. If any assertion fails, stop and return the traceback.


In [12]:
# 2.3.12 — AUDIT EXECUTION CHECKPOINT; validate and STOP before any export
p23v2c, p23v2f = part23_v2_candidates, part23_v2_firms
assert len(p23v2f) == p23v2f.gvkey.nunique() == 652
assert len(p23v2c) == 854 and not p23v2c.duplicated(["gvkey", "rcid"]).any()
assert list(p23v2f.gvkey) == list(part23_v1_firms.gvkey)
assert p23pairs(p23v2c) == p23pairs(part23_inputs["candidates"])
pd.testing.assert_series_equal(p23v2f.candidate_rcids, part23_v1_firms.candidate_rcids)
pd.testing.assert_frame_equal(
    p23v2c[p23_original_columns], part23_inputs["candidates"][p23_original_columns], check_dtype=False
)
assert matching_workflow.state_digest(part23_inputs) == part23_inputs_digest
assert matching_workflow.state_digest(part23_state) == part23_state_digest
assert matching_workflow.state_digest({"c": part23_v1_candidates, "f": part23_v1_firms}) == part23_v1_digest
pd.testing.assert_frame_equal(part23_candidate_review, part23_v1_candidates)
pd.testing.assert_frame_equal(part23_firm_review, part23_v1_firms)
pd.testing.assert_frame_equal(part23_evidence_review, part23_inputs["evidence"])
for field in ("evidence_ids", "evidence_methods", "evidence_origins", "supporting_identifier_types",
              "historical_only", "identifier_conflict_flag", "rcid_shared_across_targets"):
    pd.testing.assert_series_equal(p23v2c[field], part23_v1_candidates[field])
for frame in (p23v2c, p23v2f):
    assert frame.accepted_rcid.isna().all()
    assert frame.human_review_status.eq("unreviewed").all()
    assert frame.historical_validity_status.eq("unverified_snapshot_association").all()
    assert frame.independent_external_verification_status.eq("not_performed").all()
assert len(part23_example_ids) == len(part23_audit_example_evidence)
assert set(part23_audit_example_evidence.evidence_id) == part23_example_ids
part23_v2_leaders = p23v2f.loc[p23v2f.provisional_leading_rcid.notna()]
assert part23_v2_leaders.candidate_count.eq(1).all()
assert part23_v2_leaders.review_category.eq("Strongly corroborated candidate").all()
assert all(r.provisional_leading_rcid in r.candidate_rcids for r in part23_v2_leaders.itertuples())
assert not part23_v2_priority.duplicated(["gvkey", "rcid"]).any()
assert p23pairs(p23v2c.loc[
    p23v2c.multiple_rcid_flag | p23v2c.historical_only | p23v2c.rcid_shared_across_targets
    | p23v2c.identifier_conflict_flag | p23v2c.cusip_security_difference
]) <= p23pairs(part23_v2_priority.loc[part23_v2_priority.record_type.eq("candidate")])
assert set(part23_v2_priority.loc[part23_v2_priority.record_type.eq("unmatched_firm"), "gvkey"]) == {"179700"}
# The actual-data change must be explained entirely by removing the hierarchy veto.
part23_changed = part23_v1_candidates.review_category.ne(p23v2c.review_category)
assert part23_v1_candidates.loc[part23_changed, "review_category"].eq("Plausible candidate requiring review").all()
assert p23v2c.loc[part23_changed, "review_category"].eq("Strongly corroborated candidate").all()
assert part23_v1_candidates.loc[part23_changed, "hierarchy_scope_review"].all()
assert part23_other_gates_pass.loc[part23_changed].all()
# These observed baseline expectations detect unintended changes; the classifier does not use them.
part23_audit_validation = pd.DataFrame([
    ("firms", len(p23v2f), 652), ("pairs", len(p23v2c), 854),
    ("full_reference_pairs", int((~p23v2c.historical_only).sum()), 780),
    ("historical_only_pairs", int(p23v2c.historical_only.sum()), 74),
    ("multiple_rcid_firms", int(p23v2f.multiple_rcid_flag.sum()), 151),
    ("shared_distinct_rcids", p23v2c.loc[p23v2c.rcid_shared_across_targets, "rcid"].nunique(), 1),
    ("identifier_conflict_firms", p23v2c.loc[p23v2c.identifier_conflict_flag, "gvkey"].nunique(), 87),
    ("CUSIP_security_difference_pairs", int(p23v2c.cusip_security_difference.sum()), 135),
    ("hierarchy_metadata_inconsistency_pairs", int(p23v2c.hierarchy_metadata_inconsistency.sum()), 0),
    ("changed_categories_explained_by_hierarchy", int(part23_changed.sum()), 469),
    ("v2_provisional_leaders", len(part23_v2_leaders), 471),
    ("accepted_identities", int(p23v2f.accepted_rcid.notna().sum()), 0),
], columns=["check", "observed", "expected"])
assert part23_audit_validation.observed.eq(part23_audit_validation.expected).all()
part23_audit_validation["status"] = "PASS"
part23_v2_meeting_summary = f"""# Provisional Revelio firm-matching review
The validated universe contains 652 target GVKEY firms, 651 with candidates,
and all 854 candidate pairs: 780 full-reference and 74 historical-only pairs
across 44 firms. GVKEY 179700 remains unmatched.

Exact GVKEY, normalized CIK and validated full-CUSIP methods corroborate candidates
within retained Revelio records. Multiple methods are not independent verification.

The interpretability audit corrected a hierarchy-metadata veto: ordinary parent
or child identifiers do not demonstrate an identity conflict. Of 471 singleton
candidates passing the other conditions, the old rule excluded 469, leaving two
leaders. The corrected rule yields {len(part23_v2_leaders)} provisional leaders.
There are 471 strongly corroborated, 4 plausible, 88 ambiguous multiple-RCID,
88 conflicting-evidence, and 1 unmatched firm. These are evidence categories.

Review groups overlap: 151 firms have multiple RCIDs; 87 have GVKEY/CIK conflicts;
one distinct RCID is shared by two target GVKEYs; and 100 firms have unresolved
CUSIP security differences (135 pairs). There are 80 non-comparable CUSIP pairs,
with no observed invalid or 8/9-format cases. Historical security changes are
not established by these records. Different recorded parents remain explicit
scope-research tasks, including for 25 provisional leaders.

Illustrative concerns include Hess/Hydraulic Elements ticker overlap,
Franklin Templeton CIK associations with Digidesign/Divi Hotels, and the shared
Fox RCID. Names and identifiers raise questions, not verified corporate histories.

Every accepted_rcid remains null and all 652 firms are unreviewed. Snapshot
corroboration, annual source dates, and hierarchy metadata do not establish
2018–2025 entity continuity, ownership, or the correct historical workforce scope.
External dated research and documented human review are still required.

Next: validate this audit, then export the provisional review tables and evidence
links to a new versioned directory. Review alternatives/conflicts, validate scope
and effective dates, and record human decisions separately. No acceptance is automated.
"""
part23_v2_export_plan = pd.DataFrame([
    ("firm_review.csv", len(p23v2f)),
    ("candidate_review.csv", len(p23v2c)),
    ("review_priority.csv", len(part23_v2_priority)),
    ("meeting_summary.md", pd.NA),
], columns=["planned_file", "rows"])
display(part23_audit_validation)
display(part23_v2_export_plan)
print(part23_v2_meeting_summary)
print("PASS: Part 2.3 v2 audit, preservation, traceability and no-acceptance checks.")
print("STOP: return 2.3.11 summaries/examples and 2.3.12 validation before final exports.")
print("Files exported: 0. No reference scan or identity acceptance performed.")


,check,observed,expected,status
0,firms,652,652,PASS
1,pairs,854,854,PASS
2,full_reference_pairs,780,780,PASS
3,historical_only_pairs,74,74,PASS
4,multiple_rcid_firms,151,151,PASS
5,shared_distinct_rcids,1,1,PASS
6,identifier_conflict_firms,87,87,PASS
7,CUSIP_security_difference_pairs,135,135,PASS
8,hierarchy_metadata_inconsistency_pairs,0,0,PASS
9,changed_categories_explained_by_hierarchy,469,469,PASS


,planned_file,rows
0,firm_review.csv,652
1,candidate_review.csv,854
2,review_priority.csv,416
3,meeting_summary.md,<NA>


# Provisional Revelio firm-matching review
The validated universe contains 652 target GVKEY firms, 651 with candidates,
and all 854 candidate pairs: 780 full-reference and 74 historical-only pairs
across 44 firms. GVKEY 179700 remains unmatched.

Exact GVKEY, normalized CIK and validated full-CUSIP methods corroborate candidates
within retained Revelio records. Multiple methods are not independent verification.

The interpretability audit corrected a hierarchy-metadata veto: ordinary parent
or child identifiers do not demonstrate an identity conflict. Of 471 singleton
candidates passing the other conditions, the old rule excluded 469, leaving two
leaders. The corrected rule yields 471 provisional leaders.
There are 471 strongly corroborated, 4 plausible, 88 ambiguous multiple-RCID,
88 conflicting-evidence, and 1 unmatched firm. These are evidence categories.

Review groups overlap: 151 firms have multiple RCIDs; 87 have GVKEY/CIK conflicts;
one distinct RCID is shared by two target GVK

### 2.4 Provisional deliverables — awaiting the v2 audit checkpoint

Use the **v2** tables after the researcher validates 2.3.12:
`part23_v2_firms` (652 rows), `part23_v2_candidates` (854 rows),
`part23_v2_priority`, and `part23_v2_meeting_summary`.
The original v1 tables and executed outputs remain available for the audit trail.
`part23_evidence_review` is unchanged; `part23_audit_examples` and
`part23_audit_example_evidence` link the illustrative cases to raw evidence.

The next iteration will export the four requested deliverables into a new
`review_v2_<UTC timestamp>` directory under `data/clean/revelio_matching/`,
created exclusively with `exist_ok=False`. Evidence links, rule definitions,
source checkpoint checksums, and validation results will accompany the package.
Identifiers remain strings; nested values use documented JSON serialization.
Read CSV identifiers explicitly as strings to preserve leading zeros.

**No export runs at this checkpoint.** Existing sources, the validated evidence
checkpoint, previous outputs, and matching algorithms are preserved. No remote
upload or push is part of this workflow.
